# Agentic Multimodal Large Language Models for Moral Emotion Recognition - Experiments 1 to 9 and the Exp 16 Control

# Environment
## Packages and Settings

In [ ]:
# Experiments 1-9 and manuscript Exp 16 (labelled Experiment 14 in this notebook). Prompt templates and per-experiment procedures: Appendix C.1 (Exp 1-9) and Appendix D.1 (Exp 16); prompt knowledge and feature extraction: Appendix B; base-model selection: Methods 2.2.
import sys, os
sys.path.append(os.path.abspath("../env/configs")) 
from configs import *
list_users = list(range(1, 120))
group_names = {1: "ELVT", 2: "SYMP", 3: "INSP", 4: "CTRL"}

run_experiments = False

## Template

In [ ]:
part_first_image_only = """

The first attached video is a behavioral recording captured during an experimental procedure in which an individual is watching a short film.

The individual located at the center of the frame is the primary participant and the only subject to be analyzed.

You must make a clear and definite judgment on two emotional dimensions.

IMPORTANT: You should only use image-based information. You must not analyze, infer, or use the semantic content of the participant’s speech content nor the non-semantic acoustic features (e.g., tone, pitch, intensity, rhythm) . Consider only the image-based information.

This constraint is strict and non-negotiable.

"""

part_first_part_with_video = """

The first attached video is a behavioral recording captured during an experimental procedure in which an individual is watching a short film.

The individual located at the center of the frame is the primary participant and the only subject to be analyzed. No other individuals should be considered.

You must provide a clear, unambiguous, and definite judgment on two specified emotional dimensions. Uncertainty or vague responses are not allowed.

"""

part_main = """
A numeric score is required for each dimension in all cases.

Emotional Dimensions:

Sympathy
Inspiration

You may use the following information:

=== Extra Information ===

[[COT]]
[[KNOWLEDGE]]
[[TOOLS]]

=== Scoring Rules (Mandatory) ===

* Assign an integer score from 1 to 10 for each dimension.
* You must output a score for both sympathy and inspiration in every case.
* The scores must reflect the strength of observable emotional indicators and be reasonably distributed across the 1–10 range.
* If emotional cues are weak, ambiguous, or minimal, assign a low score (e.g., 1–3).
* If no clear emotional response is observable, assign a score of 1.
* Do not leave any field blank. Do not omit any score.

=== Output Format (Strict) ===

Output exactly in the following format:

(sympathy, inspiration) = (X, Y)

Z

"""

last_part = """
Where:

X is an integer from 1 to 10 representing sympathy.

Y is an integer from 1 to 10 representing inspiration.

Z is a complete explanation supporting the assigned scores.

=== Hard Constraints ===

* Analyze only the primary participant.
* Do not reference or repeat the prompt.
* If no video file is attached, return exactly:
  (sympathy, inspiration) = (-1, -1)
* Respond in English only.
* Ensure the output format is strictly followed.
* If JSON is required by downstream processing, ensure it is valid and properly formatted.

"""

template_main_25 = part_first_part_with_video + part_main + last_part

# Experiment 1 - Video Baseline

## Definition

In [ ]:
experiment_helper_1 = experiment_helper.copy()
experiment_helper_1.define_experiment(
    participant_sn_range= list_users,
    path_pickle_df_video= os.path.join(path_paper, "temp", "pickle", f"video_25.plk"),
    path_pickle_df_experiment_results=os.path.join(path_paper, "temp", "pickle", f"df_experiment_1.plk"),
    path_compressed_video= os.path.join(path_paper, "temp", "compressed_video"),
    experiment_step=2,
    video_only=True,
    template_main= template_main_25,
    interpreters = [],
    template_cot = "",
    template_knowledge_injection= "",
    template_tools= "",
)


In [ ]:
df_video_23 = experiment_helper_1.get_df_video_path()
with pd.option_context(
    "display.max_colwidth", None,      # show full cell content
    "display.max_rows", None,          # show all rows
    "display.max_columns", None,       # show all columns
    "display.width", None,             # auto-detect display width
    "display.expand_frame_repr", False # prevent column wrapping
):
    display(df_video_23)

## Generate Predictions

In [ ]:
run_experiments = True

if run_experiments:
    df_result_23_1 = experiment_helper_1.run_exp_moral_elevation(over_ride_existing=False,    max_workers = 3)
else:
    df_result_23_1 = experiment_helper_1.get_df_experiment_results()

In [ ]:
df_result_23_1.to_csv("df_results_25_1.csv")

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(df_result_23_1[["Group Name","Participant SN", "sympathy","inspiration","LLM_Result"]])

## Print Out Results

In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_23_1],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Exp 1. Baseline",
    font_size = 20
)

In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_23_1],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Exp 1. Baseline",
    font_size = 20
)

# Experiment 2 - Video + Audio

## Definition

In [ ]:
part_first_no_speech_text = """

The first attached video is a behavioral recording captured during an experimental procedure in which an individual is watching a short film.

The individual located at the center of the frame is the primary participant and the only subject to be analyzed.

You must make a clear and definite judgment on two emotional dimensions.

IMPORTANT: You must not analyze, infer, or use the semantic content of the participant’s speech under any circumstances. Only non-semantic acoustic features (e.g., tone, pitch, intensity, rhythm) may be considered.

This constraint is strict and non-negotiable.

"""

template_no_speech_text = part_first_no_speech_text + part_main + last_part

In [ ]:
experiment_helper_2 = experiment_helper.copy()
experiment_helper_2.define_experiment(
    participant_sn_range= list_users,
    path_pickle_df_video= os.path.join(path_paper, "temp", "pickle", f"merged_video_25.plk"),
    path_pickle_df_experiment_results=os.path.join(path_paper, "temp", "pickle", f"df_experiment_2.plk"),
    path_compressed_video= os.path.join(path_paper, "temp", "compressed_video"),
    experiment_step= 2,
    video_only=False,
    interpreters=[],
    template_main= template_no_speech_text,
    template_cot = "",
    template_knowledge_injection= "",
    template_tools= ""
)

In [ ]:
experiment_helper_2.get_df_video_path()

## Generate Predictions

In [ ]:
if run_experiments:
    df_result_25_2 = experiment_helper_2.run_exp_moral_elevation(over_ride_existing=False,    max_workers = 3)
else:
    df_result_25_2 = experiment_helper_2.get_df_experiment_results()

In [ ]:
df_result_25_2.to_csv("experiment_2.csv")

In [ ]:
df_result_25_2 = experiment_helper_2.segment_llm_response(df_result_25_2)

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(df_result_25_2[["Group Name","Participant SN", "sympathy","inspiration","LLM_Result"]])

## Print Out Results

In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_25_2],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Exp 2. V+A",
    font_size = 20
)

In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_25_2],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Exp 2. V+A",
    font_size = 20
)

# Experiment 3 - Video + Audio + Speech Context

## Definition

In [ ]:
experiment_helper_3 = experiment_helper.copy()
experiment_helper_3.define_experiment(
    participant_sn_range= list_users,
    path_pickle_df_video= os.path.join(path_paper, "temp", "pickle", f"merged_video_25.plk"),
    path_pickle_df_experiment_results=os.path.join(path_paper, "temp", "pickle", f"df_experiment_3.plk"),
    path_compressed_video= os.path.join(path_paper, "temp", "compressed_video"),
    experiment_step= 2,
    video_only=False,
    interpreters=[],
    template_main= template_main_25,
    template_cot = "",
    template_knowledge_injection= "",
    template_tools= ""
)

In [ ]:
df_video_25 = experiment_helper_3.get_df_video_path()
df_video_25

In [ ]:
df_result_25_3.to_csv("df_results_25_3.csv")

In [ ]:
df_result_25_3 = experiment_helper_3.get_df_experiment_results()
df_result_25_3


In [ ]:
with pd.option_context(
    "display.max_colwidth", None,      # show full cell content
    "display.max_rows", None,          # show all rows
    "display.max_columns", None,       # show all columns
    "display.width", None,             # auto-detect display width
    "display.expand_frame_repr", False # prevent column wrapping
):
    display(df_video_25)

## Generate Predictions

In [ ]:
if run_experiments:
    df_result_25_3 = experiment_helper_3.run_exp_moral_elevation(over_ride_existing=False,    max_workers = 3)
else:
    df_result_25_3 = experiment_helper_3.get_df_experiment_results()


In [ ]:
with pd.option_context(
    "display.max_colwidth", None,      # show full cell content
    "display.max_rows", None,          # show all rows
    "display.max_columns", None,       # show all columns
    "display.width", None,             # auto-detect display width
    "display.expand_frame_repr", False # prevent column wrapping
):
    display(df_result_25_3[["Participant SN","uploaded_file_video_merged"]])

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(df_result_25_3[["Group Name","sympathy","inspiration","LLM_Result"]])

## Print Out Results

In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_25_3],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Exp 3. V+A+T",
    font_size = 20
)

In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_25_3],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Exp 3. V+A+T",
    font_size = 20
)

In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_25_3],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Experiment 3",
    font_size = 20
)

# Experiment 6 - Knowledge Injection

## Definition

In [ ]:
template_knowledge_injection = """

The following information provides psychological guidance to help interpret the emotional reactions of the primary participant in the video.

These descriptions are conceptual references rather than strict rules. Use them only when supported by observable behavioral evidence.

========================
Emotional State Definitions
===========================

1. Sympathy

Key psychological characteristics:

* Sympathy is defined as the emotional response of the primary participant when perceiving that another person is in need of help or support.
* It is an other-oriented prosocial emotional state characterized by empathic concern for someone experiencing hardship, suffering, or vulnerability.
* It involves emotional softening and a motivation to care for, help, or support the other person.
* Core components include emotional concern, empathic understanding, and affiliative or caregiving motivation.

Typical observable indicators may include:

* Facial expressions may show concern or compassion, such as slightly drawn or inward eyebrows, a softened gaze, subtle downward tension around the mouth, or a serious and compassionate expression.
* Head movement and posture may include a slight forward or lateral head tilt, a lowered head position, or stillness, indicating attentiveness and empathic engagement.
* Eye behavior may involve sustained gaze directed toward emotionally meaningful cues or a person in distress, reflecting attentional focus and emotional engagement.
* Acoustic features may include softer, calmer, or more controlled vocal delivery, with reduced intensity, slower pace, and a tone conveying warmth, seriousness, or concern.
* Speech content, if permitted, may include expressions acknowledging another person’s hardship or conveying empathy, care, or emotional support.

2. Inspiration

Key psychological characteristics:

* Inspiration is defined as the emotional response of the primary participant when observing behaviors that are admirable.
* Admirable behaviors include actions that are difficult to achieve or prosocial acts such as helping others, caring for family members, or making meaningful sacrifices.
* This state is associated with admiration, motivation, uplift, and positive emotional arousal.
* Core components include increased motivation, aspiration, cognitive engagement with meaning or value, and heightened attention.

Typical observable indicators may include:

* Facial expressions may show engaged positive attention, such as subtle smiling, raised cheeks, slightly lifted eyebrows, widened or attentive eyes, or expressions reflecting admiration or engaged attention.
* Head movement and posture may appear upright or slightly elevated, with an open posture and orientation toward the observed action. Stillness or slight backward head movement may indicate being impressed.
* Eye behavior may involve sustained and focused visual attention, often with longer fixation, indicating cognitive processing of meaningful or admirable behavior.
* Acoustic features may include more animated or expressive vocal delivery, with higher energy, greater pitch variation, and dynamic rhythm.
* Speech content, if permitted, may include expressions of admiration, praise, or reflections about meaning, value, or a desire to emulate the observed behavior.

"""

In [ ]:
experiment_helper_6 = experiment_helper.copy()
experiment_helper_6.define_experiment(
    participant_sn_range= list_users,
    path_pickle_df_video= os.path.join(path_paper, "temp", "pickle", f"merged_video_25.plk"),
    path_pickle_df_experiment_results=os.path.join(path_paper, "temp", "pickle", f"df_experiment_6.plk"),
    path_compressed_video= os.path.join(path_paper, "temp", "compressed_video"),
    experiment_step= 25,
    video_only=False,
    interpreters=[],
    template_main= template_main_25,
    template_cot = "",
    template_knowledge_injection= template_knowledge_injection,
    template_tools= "",
)

## Generate Predictions

In [ ]:
if run_experiments:
    df_result_25_6 = experiment_helper_6.run_exp_moral_elevation(over_ride_existing=False,    max_workers = 3)
else:
    df_result_25_6 = experiment_helper_6.get_df_experiment_results()

In [ ]:
df_result_25_6.to_csv("experiment_6.csv")

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(df_result_25_6[["Group Name","sympathy","inspiration","LLM_Result"]])

## Print Out Results

In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_25_6],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Exp 6. Know. Inject.",
    font_size = 20
)

In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_25_6],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Exp 6. Know. Inject.",
    font_size = 20
)

# Experiment 5 - CoT

## Definition

In [ ]:

template_cot = """
[Important]

You must reason step by step before assigning scores.

Step 1. Definition of Emotional Constructs: sympathy and inspiration.

Sympathy is the emotional response of the primary participant when perceiving that another person is in need of help or support. Inspiration is the emotional response of the primary participant when observing behaviors that are admirable, such as performing exceptional actions or helping others.

Generate comprehensive definitions of both sympathy and inspiration before proceeding.

Step 2. Theoretical Multimodal Profile

* Briefly describe the typical behavioral and physiological indicators of (a) sympathy and (b) inspiration.

* Your description must include, but is not limited to, the following modalities:

(1) Facial expression, including relevant Facial Action Units (AUs) and overall facial configuration.

(2) Head posture and head movement dynamics.

(3) Eye behavior, including gaze direction, fixation duration, and blink rate.

(4) Acoustic or speech features, including vocal qualities such as pitch, intensity, and tempo.

(5) Speech content, if permitted by the constraints.

Step 3. Cross-Modal Integration

* Compare the observed behavioral patterns with the theoretical profiles.

* Determine which emotional pattern (sympathy or inspiration) is better supported.

* Justify your conclusion based on convergence across multiple modalities rather than relying on a single cue.

Step 4. Final Scoring

* After completing the reasoning above, assign integer scores from 1 to 10 for both sympathy and inspiration according to the scoring rules.

* Do not skip any step.

"""

In [ ]:
experiment_helper_5 = experiment_helper.copy()
experiment_helper_5.define_experiment(
    participant_sn_range= list_users,
    path_pickle_df_video= os.path.join(path_paper, "temp", "pickle", f"merged_video_25.plk"),
    path_pickle_df_experiment_results=os.path.join(path_paper, "temp", "pickle", f"df_experiment_5.plk"),
    path_compressed_video= os.path.join(path_paper, "temp", "compressed_video"),
    experiment_step= 25,
    video_only=False,
    interpreters=[],
    template_main=template_main_25,
    template_cot = template_cot,
    template_knowledge_injection= "",
    template_tools= "",
)

## Generate Predictions

In [ ]:
if run_experiments:
    df_result_25_5 = experiment_helper_5.run_exp_moral_elevation(over_ride_existing=False,    max_workers = 3)
else:
    df_result_25_5 = experiment_helper_5.get_df_experiment_results()
    df_result_25_5.to_csv("experiment_5.csv")

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(df_result_25_5[["Group Name","sympathy","inspiration","LLM_Result"]])

## Print Out Results

In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_25_5],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Exp 5. CoT",
    font_size = 20
)

In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_25_5],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="E.5 CoT",
    font_size = 20
)

# Experiment 7 - Few-Shot Learning

## Definition

In [ ]:
part_first_few_shot_learning = """

The first attached video is a behavioral recording captured during an experimental procedure.

The individual located at the center of the frame is the primary participant and the only subject to be analyzed.

You must make a clear and definite judgment on two emotional dimensions.

Use the information provided from other videos as few-shot examples to guide your analysis of the current video. These examples are for reference only. You must determine the scores based on your own analysis of the current video, not by copying or relying directly on the examples. Some examples may not be relevant and should be disregarded if they do not apply.

"""

part_last_few_shot_learning = """

=== Further Reminder ===

Z must explicitly explain how the provided few-shot examples informed your analysis of the current video. The explanation should clarify which aspects of the examples were relevant, how they guided your interpretation, and why they were applicable or not applicable to the current case.

"""

template_main_25_few_shot = part_first_few_shot_learning + part_main + part_last_few_shot_learning

num_folds = 5

import google.generativeai as genai
genai.configure(api_key=env_config.api_key_gemini)

folds = experiment_helper.generate_random_folds(list_users, num_folds)

df_video_23 = experiment_helper_2.get_df_video_path()
df_result_25_2 = experiment_helper_2.get_df_experiment_results()

experiment_helper_7 = experiment_helper.copy()
experiment_helper_7.define_experiment(
    participant_sn_range= list_users,
    path_pickle_df_video= os.path.join(path_paper, "temp", "pickle", "merged_video_25.plk"),
    path_pickle_df_experiment_results=os.path.join(path_paper, "temp", "pickle", "df_experiment_7.plk"),
    experiment_step=25,
    video_only=False,
    template_main=template_main_25,
    interpreters = [],
    template_cot = "",
    template_knowledge_injection= "",
    template_tools= "",
    verbose = True,
    path_compressed_video= os.path.join(path_paper, "temp", "compressed_video"),
)

## Data Preprocessing

In [ ]:
import pandas as pd


def get_expiration(file_obj):
    """
    Safely get expiration date from File object.
    """
    if file_obj is None or pd.isna(file_obj):
        return None

    for attr in ["expiration_time", "expires_at", "expiration", "expiry"]:
        if hasattr(file_obj, attr):
            return getattr(file_obj, attr)

    return None


cols = ["uploaded_file_video_only", "uploaded_file_video_merged"]


for idx, row in df_result_25_3.iterrows():

    sn = row["Participant SN"]

    match = df_video_25[df_video_25["Participant SN"] == sn]

    if match.empty:
        continue

    new_row = match.iloc[0]

    for col in cols:

        old_file = row[col]
        new_file = new_row[col]

        # If new has no value, skip
        if new_file is None or pd.isna(new_file):
            continue

        # If old has no value, replace directly
        if old_file is None or pd.isna(old_file):

            print(f"Participant SN {sn} column {col} replaced (old empty → new used)")
            col_idx = df_result_25_3.columns.get_loc(col)
            df_result_25_3.iat[idx, col_idx] = new_file
            # df_result_25_3.at[idx, col] = new_file
            continue

        # Both have values, compare expiration
        exp_old = get_expiration(old_file)
        exp_new = get_expiration(new_file)

        if exp_old is None or exp_new is None:
            continue

        if exp_new > exp_old:

            print(
                f"Participant SN {sn} column {col} replaced "
                f"{exp_old} → {exp_new}"
            )
            # Use iat (integer-based position) instead of at (label-based)
            # You need the integer index for the column:
            col_idx = df_result_25_3.columns.get_loc(col)
            df_result_25_3.iat[idx, col_idx] = new_file


            # df_result_25_3.at[idx, col] = new_file

## Generate Predictions

In [ ]:
df_result_25_3 = experiment_helper_7.segment_llm_response(df_result_25_3)

In [ ]:
df_result_25_3["explanation"]

In [ ]:
# run_experiments = True

In [ ]:

if run_experiments:
    num_folds = 5
    df_result_25_7 = experiment_helper_7.run_few_shot_learning(
        df_video_info = df_result_25_3, 
        folds=folds, 
        num_folds=num_folds, 
        genai = genai, 
        template_main=template_main_25_few_shot,
        experiment_helper = experiment_helper,
        video_only=False,
        similarity_based_sampling=True,
        similarity_based_sampling_by_group = True,
        over_ride_existing=False,
        max_workers=3,
        number_of_few_shot_examples = 3,
        verbose = False
    )
else:
    df_result_25_7 = experiment_helper_7.get_df_experiment_results()
    df_result_25_7.to_csv("experiment_7.csv")

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(df_result_25_7[["Group","Group Name","sympathy","inspiration","LLM_Result"]])

## Print Out Results

In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_25_7],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Exp 7. Few-Shot",
    font_size = 20
)

In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_25_7],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Experiment 6",
    font_size = 20
)

# Experiment 8 - Fine-tuning

## Definition


In [ ]:
bucket_name = "llm_moral_emotion"
path_to_sa_key = os.path.join(env_config.path_others_cache, "credentials","gcs_vertex", "llm-1-439114-326732d42ef9.json")
scopes = [ 'https://www.googleapis.com/auth/cloud-platform']
bucket_uri = f"gs://{bucket_name}"
project_id = "llm-1-439114"
region = "us-west1"
base_model = "gemini-2.5-flash"
num_folds = 5 # Split into 5 folds

try:
    # 1. Load credentials from file information and specify SCOPES
    # This is a method to bypass os.environ and DefaultCredentialsError
    credentials = service_account.Credentials.from_service_account_file(
        path_to_sa_key,
        scopes=scopes  # Fix RefreshError
    )
    
    # 2. Initialize GenAI client, explicitly pass credentials
    client = genai.Client(
        credentials=credentials,
        vertexai=True, 
        project=project_id, 
        location=region
    )
    
    print("✅ GenAI client successfully initialized with the manually-loaded service account key and correct scope.")

except FileNotFoundError:
    print(f"❌ Error: Key file not found. Please verify the path is correct:{PATH_TO_SA_KEY}")
except Exception as e:
    print(f"❌ Authentication or initialization failed:{e}")


gcs_storage_helper = StorageHelper(
    project_id = project_id,
    region = region ,
    bucket_name = bucket_name,
    target_prefix = "videos/" ,  # Target subdirectory (files will be uploaded to gs://TARGET_BUCKET_NAME/TARGET_PREFIX/...)
    video_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.wmv'), # Supported video file extensions (lowercase)
    path_to_sa_key = path_to_sa_key
)

psychai_rouge_evaluator = PsychAIRougeEvaluator(tasks=["rouge1", "rouge2", "rougeL"])

In [ ]:
# ==========================================================
# Basic Configuration
# ==========================================================

# Standardize all user IDs to 3-digit strings with leading zeros.
# For example:
#   1   → "001"
#   12  → "012"
#   123 → "123"
#
# This ensures consistency with the "Participant SN" column in the data table,
# Especially when the field is in string format, this can avoid comparison errors.
formatted_list_users = [str(u).zfill(3) for u in list_users]

import shutil
path_pickle_df_video=os.path.join(path_paper, "temp", "pickle", "merged_video_25.plk")
path_compressed_video = os.path.join(path_paper, "temp", "compressed_video"),
path_pickle_df_experiment_results_results_2 = os.path.join(path_paper, "temp", "pickle", "df_experiment_3.plk")
path_pickle_df_experiment_results=os.path.join(path_paper, "temp", "pickle", "df_25_exp_3_copy_for_exp_8.plk")
if not os.path.exists(path_pickle_df_experiment_results):
    shutil.copy2(path_pickle_df_experiment_results_results_2, path_pickle_df_experiment_results)

template_path_pickle_df_experiment_results=os.path.join(path_paper, "temp", "pickle", "df_25_exp_8_{i}.plk"),
experiment_helper_8 = experiment_helper.copy()
experiment_helper_8.define_experiment(
    participant_sn_range= list_users,
    path_pickle_df_video= path_pickle_df_video,
    path_pickle_df_experiment_results=path_pickle_df_experiment_results,
    path_compressed_video= path_compressed_video,
    experiment_step=25,
    video_only=False,
    template_main=template_main_25,
    interpreters = [],
    template_cot = "",
    template_knowledge_injection= "",
    template_tools= "",
    verbose = True,
)

In [ ]:


fine_tuning_helper = FineTuningHelper(gcs_storage_helper, experiment_helper)

df_result_25_8 = experiment_helper_8.get_df_experiment_results()

df_result_25_8 = fine_tuning_helper.update_ground_truth_column(df_result_25_8, llm_client, verbose=True)

# df_experiment_before_llm_updated.to_pickle(path_pickle_df_experiment_results)

folds = experiment_helper_8.generate_random_folds(list_users, num_folds)

# df_experiment_before_llm_updated.head()

In [ ]:
with pd.option_context(
    "display.max_colwidth", None,      # show full cell content
    "display.max_rows", None,          # show all rows
    "display.max_columns", None,       # show all columns
    "display.width", None,             # auto-detect display width
    "display.expand_frame_repr", False # prevent column wrapping
):
    display(df_result_25_8[["Participant SN","Group Name", "sympathy","inspiration","LLM_Result", "explanation", "ground_truth"]])

### Step 1: Dataset Preparation (Run Once)

In [ ]:
# ==========================================================
# Step 1: Dataset preparation (execute once only)
# ==========================================================
# Purpose:
# - Create training and validation sets for each fold
# - Convert to JSONL format
# - Upload JSONL file to remote LLM service
# - Store intermediate information required for subsequent training loop
#
# Important Note:
# This step is responsible for "data preparation" only.
# Model training or prediction has not yet been performed.
#
# Why separate data preparation from training?
# - Avoid generating duplicate JSONL files
# - Make fine-tuning loop structure clearer
# - Improve experimental reproducibility and controllability
# ==========================================================

import os
import pickle

# ----------------------------------------------------------
# fold_datasets pickle path
# ----------------------------------------------------------
# path_fold_datasets_pickle = os.path.join(
#     path_pickle_dir, f"fold_datasets_step_{step}.pkl"
# )
path_fold_datasets_pickle = os.path.join(path_paper, "temp", "pickle",f"fold_datasets_step_25.pkl")

# ----------------------------------------------------------
# If pickle already exists, load it directly
# ----------------------------------------------------------
if os.path.exists(path_fold_datasets_pickle):

    print("✓ Found fold_datasets pickle; loading dataset information directly.")

    with open(path_fold_datasets_pickle, "rb") as f:
        fold_datasets = pickle.load(f)

    print(f"✓ Loaded {len(fold_datasets)} folds of data.")

else:

    fold_datasets = []  # Storing intermediate information for each fold

    # Process each fold sequentially
    # Assuming the folds structure is as follows:
    #   folds = [
    #       [fold 0 test user ID list],
    # [fold 1 test user ID list],
    #       ...
    #   ]
    for fold_index in range(len(folds)):

        print(f"--- Preparing dataset for Fold {fold_index+1}/{num_folds} ---")

        # ------------------------------------------------------
        # ✓ Determine training/testing split for this fold
        # ------------------------------------------------------

        # This fold contains users for validation/testing purposes
        test_users = folds[fold_index]

        # Training data source: all users not in test_users
        train_pool = [u for u in formatted_list_users if u not in test_users]


        # ------------------------------------------------------
        # 2️⃣ Setting JSONL output path
        # ------------------------------------------------------
        # Each fold uses independent JSONL files, ensuring:
        # - will not overwrite each other
        #   - Clearly traceable experiment source
        path_jsonl_training = f"./jsonl/train_{step}_fold_{fold_index}.jsonl"
        path_jsonl_validation = f"./jsonl/validation_{step}_fold_{fold_index}.jsonl"


        # ------------------------------------------------------
        # 3️⃣ Building training data DataFrame
        # ------------------------------------------------------
        # Filter data rows where Participant SN belongs to train_pool.
        # Reset index, avoiding retention of the original DataFrame's index.
        df_training = (
            df_result_25_8
            .loc[df_result_25_8["Participant SN"].isin(train_pool)]
            .reset_index(drop=True)
        )


        # ------------------------------------------------------
        # 4️⃣ Create validation data DataFrame
        # ------------------------------------------------------
        # Keep only data rows belonging to test_users.
        df_validation = (
            df_result_25_8
            .loc[df_result_25_8["Participant SN"].isin(test_users)]
            .reset_index(drop=True)
        )


        # ------------------------------------------------------
        # 5️⃣ Convert training data to JSONL and upload
        # ------------------------------------------------------
        # fine_tuning_data_preparation will:
        # - Organize data into LLM training format
        # - Generate JSONL file locally
        # - Upload to remote service
        # - Return:
        # df_training (possibly updated)
        # data_training (formatted data)
        # path_uploaded_jsonl_training (remote file reference path)
        df_training, data_training, path_uploaded_jsonl_training = (
            fine_tuning_helper.fine_tuning_data_preparation(
                df_training,
                prefix="train",
                path_jsonl=path_jsonl_training,
                llm_client=llm_client,
                verbose=True
            )
        )


        # ------------------------------------------------------
        # 6️⃣ Convert validation data to JSONL and upload
        # ------------------------------------------------------
        # On certain platforms, verification of data uploads may be optional,
        # But maintain consistent process here for ease of management.
        df_validation, data_validation, path_uploaded_jsonl_validation = (
            fine_tuning_helper.fine_tuning_data_preparation(
                df_validation,
                prefix="validation",
                path_jsonl=path_jsonl_validation,
                llm_client=llm_client,
                verbose=True
            )
        )


        # ------------------------------------------------------
        # 7️⃣ Save metadata for this Fold
        # ------------------------------------------------------
        # This location does not store df_training because:
        # - Training executed remotely
        # - Subsequent predictions only require validation data
        #
        # The following dictionary contains all the information required for subsequent fine-tuning and prediction.
        fold_datasets.append({
            "fold_index": fold_index,
            "df_validation": df_validation,
            "path_uploaded_jsonl_training": path_uploaded_jsonl_training,
            "path_uploaded_jsonl_validation": path_uploaded_jsonl_validation
        })

        # ------------------------------------------------------
        # Save pickle after each fold completed (supports resuming from interruption)
        # ------------------------------------------------------
        with open(path_fold_datasets_pickle, "wb") as f:
            pickle.dump(fold_datasets, f)

        print(f"✓ Fold {fold_index} data written to pickle")


# ==========================================================
# Step 1 results completed
# ==========================================================
# fold_datasets will form the following structure:
#
# [
#   {
#     "fold_index": 0,
#     "df_validation": <DataFrame>,
#     "path_uploaded_jsonl_training": "...",
#     "path_uploaded_jsonl_validation": "..."
#   },
#   {
#     "fold_index": 1,
#     ...
#   },
#   ...
# ]
#
# This data structure will be directly supplied to subsequent:
# - fine-tuning loop
# - Predict loop
# Use
# ==========================================================

### Step 2A: Model Fine-Tuning (Training Loop)

In [ ]:
# ==========================================================
# Step 2 A: Model fine-tuning (training loop)
# ==========================================================
# Purpose:
# - Train independent fine-tuned models for each fold
# - Record model_url for each fold using pickle
# - If the fold already exists in pickle, skip it directly
# - Support program interruption and continue
# ==========================================================

# ----------------------------------------------------------
# model_urls pickle path
# ----------------------------------------------------------
path_model_urls_pickle = os.path.join(path_paper, "temp", "pickle", f"model_urls_25.plk")

# ----------------------------------------------------------
# If pickle already exists, read it first
# ----------------------------------------------------------
if os.path.exists(path_model_urls_pickle):

    with open(path_model_urls_pickle, "rb") as f:
        model_urls = pickle.load(f)

    print(f"✓ Read model_urls pickle, {len(model_urls)} folds total.")

else:

    model_urls = {}
    print("⚠ model_urls pickle not found; recreating.")


# ----------------------------------------------------------
# Start training loop
# ----------------------------------------------------------
for fold_data in fold_datasets:

    fold_index = fold_data["fold_index"]

    # ------------------------------------------------------
    # If the fold is already in pickle, skip directly
    # ------------------------------------------------------
    if fold_index in model_urls:

        print(f"✓ Fold {fold_index} model URL already exists; skipping fine-tuning.")
        continue

    print(f"--- Fine-tuning Fold {fold_index+1}/{num_folds} ---")

    # ------------------------------------------------------
    # Execute Supervised Fine-Tuning
    # ------------------------------------------------------
    fine_tuned_model_url = fine_tuning_helper.fine_tune_model(
        client=client,
        base_model=base_model,
        model_name=f"llm_moral_elevation_{fold_index}",
        path_uploaded_jsonl_training=fold_data["path_uploaded_jsonl_training"],
        path_uploaded_jsonl_validation=fold_data["path_uploaded_jsonl_validation"],
        metadata_dir="../temp/tuning_metadata",
        poll_interval=60,
        retry_interval=30,
        max_retry=5,
    )

    # ------------------------------------------------------
    # Update dictionary
    # ------------------------------------------------------
    model_urls[fold_index] = fine_tuned_model_url

    print(f"✓ Fold {fold_index} model fine-tuning complete.")

    # ------------------------------------------------------
    # Save pickle immediately (avoid loss if interrupted)
    # ------------------------------------------------------
    with open(path_model_urls_pickle, "wb") as f:
        pickle.dump(model_urls, f)

    print("✓ model_urls pickle updated and saved.")


# ----------------------------------------------------------
# Final display all folds
# ----------------------------------------------------------
print("\n=== Completed Fold Models ===")
for k, v in model_urls.items():
    print(f"Fold {k} -> {v}")

### Step 2B: Prediction Loop

In [ ]:
# ==========================================================
# Step 2 B: Prediction Loop
# ==========================================================
# Purpose:
# - For each fold, use its corresponding fine-tuned model for prediction
# - Save results immediately after prediction is complete
# - Fully supports resuming after interruption (resume-safe)
#
# Design Principles:
# - Even if an error occurs partway through the prediction process, already-completed folds retain their results
# - Training and inference errors are isolated from each other and do not affect one another
# ==========================================================

all_fold_dfs = []   # Used to store results from each fold, then merge at the end

for fold_data in fold_datasets:

    fold_index = fold_data["fold_index"]

    # path_pickle_fold = os.path.join(path_pickle_dir, f"fine_tuning_prediction_fold_{fold_index}.pkl")

    path_pickle_fold = os.path.join(path_paper, "temp", "pickle",  f"fine_tuning_prediction_25_{fold_index}.pkl")
    path_pickle_temp = os.path.join(path_paper, "temp", "pickle",  f"fine_tuning_prediction_25_{fold_index}_tmp.pkl")
    
    # ---- Interruption Resume Check ----
    # If the pickle file for that fold already exists:
    # - No longer performing prediction
    # - Reading saved results directly
    # - Add result to final list
    if os.path.exists(path_pickle_fold):

        print(f"✓ Fold {fold_index} already has results; loading directly.")

        df_fold_result = pd.read_pickle(path_pickle_fold)

        all_fold_dfs.append(df_fold_result)

        continue


    print(f"--- Predicting Fold {fold_index+1}/{num_folds} ---")


    # ----------- Safety Check -----------
    # If the training loop fails or is incomplete,
    # May result in no corresponding model in model_urls.
    if fold_index not in model_urls:

        raise ValueError(
            f"Fine-tuned model for Fold {fold_index} not found. "
            "Please confirm the training loop completed successfully."
        )


    fine_tuned_model_url = model_urls[fold_index]


    # ------------------------------------------------------
    # Predicting on validation data using the fine-tuned model
    # ------------------------------------------------------
    df_fold_result = fine_tuning_helper.extend_df_with_prediction(
        client,
        fold_data["df_validation"],
        model=fine_tuned_model_url
    )


    # ------------------------------------------------------
    # Prevent chained assignment issue
    # ------------------------------------------------------
    df_fold_result = df_fold_result.copy()


    # ------------------------------------------------------
    # Add fold mark field
    # ------------------------------------------------------
    df_fold_result["fold"] = fold_index


    # ------------------------------------------------------
    # First write to temporary pickle (avoid crash corruption of file)
    # ------------------------------------------------------
    df_fold_result.to_pickle(path_pickle_temp)


    # ------------------------------------------------------
    # atomic rename → formal pickle
    # ------------------------------------------------------
    os.replace(path_pickle_temp, path_pickle_fold)


    print(f"✓ Fold {fold_index} prediction complete and saved.")


    all_fold_dfs.append(df_fold_result)


# ==========================================================
# Step 3: Merge all fold results
# ==========================================================
# After completing all folds (whether it is:
# - Load existing results from disk
#   - or generate new execution prediction results)
#
# Merge DataFrames from each fold into a single summary results table.
# ==========================================================

df_all_results_25_8 = pd.concat(all_fold_dfs, ignore_index=True)


# ----------------------------------------------------------
# Save final consolidated result
# ----------------------------------------------------------
path_pickle_df_experiment_results_final = os.path.join(path_paper, "temp", "pickle", "df_experiment_8.plk")
df_all_results_25_8.to_pickle(path_pickle_df_experiment_results_final)

print("✓ All Folds completed and successfully integrated into the final results.")

## Print Out Results

In [ ]:
path_pickle_df_experiment_results_final = os.path.join(path_paper, "temp", "pickle", "df_experiment_8.plk")
df_all_results_25_8 = pd.read_pickle(path_pickle_df_experiment_results_final)

In [ ]:
df_all_results_25_8.to_csv("experiment_8.csv")

In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_all_results_25_8],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Exp 8. Fine-Tuned",
    font_size = 20
)

# Experiment 4 - Video + Audio + Physio

## Definition

#### acoustic

In [ ]:
acoustic_criteria_list = [

  # ------------------------------------------------------------------
  # Pitch level
  # Higher pitch is associated with emotional activation and enthusiasm.
  # ------------------------------------------------------------------
  ["median_acoustics-F0semitoneFrom27.5Hz_sma3nz_amean", ">", "q3",
   "The speaker's voice pitch is noticeably higher than typical levels, which often occurs when a person speaks with enthusiasm, admiration, or emotional excitement."],

  ["median_acoustics-F0semitoneFrom27.5Hz_sma3nz_amean", "<=", "q1",
   "The speaker's voice pitch is relatively low, which can indicate calmness, subdued emotional expression, or reflective concern."],


  # ------------------------------------------------------------------
  # Pitch variability
  # Emotional speech often contains stronger pitch fluctuations.
  # ------------------------------------------------------------------
  ["median_acoustics-F0semitoneFrom27.5Hz_sma3nz_stddevNorm", ">", "q3",
   "The pitch fluctuates considerably during speech, suggesting expressive and emotionally engaged speaking."],

  ["median_acoustics-F0semitoneFrom27.5Hz_sma3nz_stddevNorm", "<=", "q1",
   "The pitch remains relatively stable and monotonic, suggesting restrained or emotionally neutral speech."],


  # ------------------------------------------------------------------
  # Loudness (vocal energy)
  # Emotionally engaged or enthusiastic speech often increases vocal energy.
  # ------------------------------------------------------------------
  ["median_acoustics-loudness_sma3_amean", ">", "q3",
   "The speaker produces stronger vocal energy than typical, which often accompanies enthusiastic or emotionally activated speech."],

  ["median_acoustics-loudness_sma3_amean", "<=", "q1",
   "The vocal energy is relatively low, suggesting quiet, gentle, or reflective emotional expression."],


  # ------------------------------------------------------------------
  # Loudness variability
  # Emotionally expressive speech often contains dynamic energy changes.
  # ------------------------------------------------------------------
  ["median_acoustics-loudness_sma3_stddevNorm", ">", "q3",
   "The loudness varies considerably throughout the speech, suggesting expressive or emotionally animated speaking."],

  ["median_acoustics-loudness_sma3_stddevNorm", "<=", "q1",
   "The loudness is relatively stable, suggesting controlled or emotionally neutral speech delivery."],


  # ------------------------------------------------------------------
  # Spectral flux
  # Reflects articulation dynamics and vocal activity.
  # ------------------------------------------------------------------
  ["median_acoustics-spectralFlux_sma3_amean", ">", "q3",
   "The spectral structure of the voice changes rapidly over time, suggesting energetic articulation and emotional engagement."],

  ["median_acoustics-spectralFlux_sma3_amean", "<=", "q1",
   "The spectral structure changes slowly, suggesting steady and less expressive articulation."],


  # ------------------------------------------------------------------
  # Jitter (frequency instability)
  # Stress or emotional intensity can increase jitter.
  # ------------------------------------------------------------------
  ["median_acoustics-jitterLocal_sma3nz_amean", ">", "q3",
   "The voice shows irregular fluctuations in pitch, suggesting unstable vocal fold vibration that may occur under emotional intensity."],

  ["median_acoustics-jitterLocal_sma3nz_amean", "<=", "q1",
   "The vocal pitch is stable with minimal irregularity, suggesting steady and controlled phonation."],


  # ------------------------------------------------------------------
  # Shimmer (amplitude instability)
  # Reflects instability in vocal intensity.
  # ------------------------------------------------------------------
  ["median_acoustics-shimmerLocaldB_sma3nz_amean", ">", "q3",
   "The amplitude of the voice fluctuates noticeably, indicating unstable vocal intensity which can occur during expressive emotional speech."],

  ["median_acoustics-shimmerLocaldB_sma3nz_amean", "<=", "q1",
   "The amplitude remains stable, suggesting controlled vocal production."],


  # ------------------------------------------------------------------
  # Harmonic-to-Noise Ratio
  # Vocal clarity reflects stability of phonation.
  # ------------------------------------------------------------------
  ["median_acoustics-HNRdBACF_sma3nz_amean", ">", "q3",
   "The voice contains strong harmonic structure and little noise, suggesting a clear and stable vocal quality."],

  ["median_acoustics-HNRdBACF_sma3nz_amean", "<=", "q1",
   "The voice contains more noise relative to harmonic components, suggesting breathy or strained phonation."],


  # ------------------------------------------------------------------
  # Formant F1
  # Reflects openness of vocal tract articulation.
  # ------------------------------------------------------------------
  ["median_acoustics-F1frequency_sma3nz_amean", ">", "q3",
   "The first formant frequency is relatively high, suggesting more open vocal articulation."],

  ["median_acoustics-F1frequency_sma3nz_amean", "<=", "q1",
   "The first formant frequency is relatively low, suggesting more closed articulation."],


  # ------------------------------------------------------------------
  # Formant F2
  # Reflects tongue position and articulation.
  # ------------------------------------------------------------------
  ["median_acoustics-F2frequency_sma3nz_amean", ">", "q3",
   "The second formant frequency is higher, suggesting forward tongue articulation."],

  ["median_acoustics-F2frequency_sma3nz_amean", "<=", "q1",
   "The second formant frequency is lower, suggesting more central or retracted articulation."],

]

### Head Position

In [ ]:
head_position_criteria_list = [

["pose-end_to_begin_ear_to_ear_difference_x", ">", "q3",
"The horizontal distance between the ears changes noticeably, suggesting lateral head movement."],

["pose-end_to_begin_ear_to_ear_difference_y", ">", "q3",
"The vertical distance between the ears changes, indicating head tilting or posture adjustment."],

["pose-end_to_begin_ear_to_ear_difference_z", ">", "q3",
"Depth change between the ears suggests forward or backward head movement."],

["pose-end_to_begin_shoulder_to_shoulder_difference_x", ">", "q3",
"The shoulders move horizontally, suggesting body posture adjustment."],

["pose-end_to_begin_shoulder_to_shoulder_difference_y", ">", "q3",
"The shoulders move vertically, indicating posture change or body shift."],

["pose-end_to_begin_shoulder_to_shoulder_difference_z", ">", "q3",
"Depth change in shoulder position suggests leaning forward or backward."],

["pose-end_to_begin_nose_to_lip_difference_x", ">", "q3",
"The relative horizontal position between nose and lips changes."],

["pose-end_to_begin_nose_to_lip_difference_y", ">", "q3",
"Vertical movement between the nose and lips suggests head tilting or speaking movement."],

["pose-end_to_begin_nose_to_lip_difference_z", ">", "q3",
"Depth change between the nose and lips suggests forward or backward head movement."],

["pose-end_to_begin_left_head_angle", ">", "q3",
"The head tilts noticeably toward the left side."],

["pose-end_to_begin_right_head_angle", ">", "q3",
"The head tilts noticeably toward the right side."],

]

### Eye Movement

In [ ]:
eye_movement_criteria_list = [

["median_eye_move-mean_left_iris_relative_x", ">", "q3",
"The left iris is positioned further horizontally, suggesting sideways gaze."],

["median_eye_move-mean_right_iris_relative_x", ">", "q3",
"The right iris shows horizontal displacement, indicating gaze shift."],

["median_eye_move-mean_left_iris_relative_y", ">", "q3",
"The left iris moves vertically, suggesting change in gaze elevation."],

["median_eye_move-mean_right_iris_relative_y", ">", "q3",
"The right iris moves vertically, indicating upward or downward gaze shifts."],

["median_eye_move-variance_left_iris_relative_x", ">", "q3",
"The horizontal position of the left iris varies substantially."],

["median_eye_move-variance_right_iris_relative_x", ">", "q3",
"The horizontal position of the right iris varies considerably."],

["median_eye_move-variance_left_iris_relative_y", ">", "q3",
"The vertical position of the left iris fluctuates noticeably."],

["median_eye_move-variance_right_iris_relative_y", ">", "q3",
"The vertical position of the right iris fluctuates noticeably."],

["median_eye_move-mean_left_eye_height", ">", "q3",
"The left eye appears more open than usual, which may indicate heightened attention."],

["median_eye_move-mean_right_eye_height", ">", "q3",
"The right eye appears more open than typical."],

["median_eye_move-variance_left_eye_height", ">", "q3",
"The left eye opening varies noticeably, suggesting blinking or eyelid movement."],

["median_eye_move-variance_right_eye_height", ">", "q3",
"The right eye opening fluctuates, indicating blinking."],

]


### Text Emotion

In [ ]:

text_emotion_criteria_list = [

["median_speech_text-admiration", ">", "q3",
"The speech frequently expresses admiration or appreciation toward something."],

["median_speech_text-amusement", ">", "q3",
"The spoken content reflects amusement or humor."],

["median_speech_text-approval", ">", "q3",
"The speech frequently conveys approval or agreement."],

["median_speech_text-caring", ">", "q3",
"The speech reflects caring or supportive emotional expressions."],

["median_speech_text-gratitude", ">", "q3",
"The spoken content expresses gratitude or thankfulness."],

["median_speech_text-joy", ">", "q3",
"The speech frequently conveys joy or happiness."],

["median_speech_text-love", ">", "q3",
"The spoken content expresses affection or love."],

["median_speech_text-optimism", ">", "q3",
"The speech reflects optimism or positive expectations."],

["median_speech_text-pride", ">", "q3",
"The speech expresses pride or a sense of accomplishment."],

["median_speech_text-relief", ">", "q3",
"The spoken content suggests relief from tension."],

["median_speech_text-anger", ">", "q3",
"The speech expresses anger or frustration."],

["median_speech_text-annoyance", ">", "q3",
"The spoken content reflects irritation."],

["median_speech_text-disapproval", ">", "q3",
"The speech expresses disapproval or criticism."],

["median_speech_text-disgust", ">", "q3",
"The spoken content reflects disgust."],

["median_speech_text-disappointment", ">", "q3",
"The speech reflects disappointment."],

["median_speech_text-embarrassment", ">", "q3",
"The speech reflects embarrassment."],

["median_speech_text-fear", ">", "q3",
"The spoken content expresses fear or anxiety."],

["median_speech_text-grief", ">", "q3",
"The speech reflects grief or sadness."],

["median_speech_text-nervousness", ">", "q3",
"The speech expresses nervousness or tension."],

["median_speech_text-remorse", ">", "q3",
"The spoken content reflects regret or remorse."],

["median_speech_text-sadness", ">", "q3",
"The speech expresses sadness."],

["median_speech_text-confusion", ">", "q3",
"The speech indicates confusion."],

["median_speech_text-curiosity", ">", "q3",
"The spoken content reflects curiosity or interest."],

["median_speech_text-desire", ">", "q3",
"The speech expresses desire or wanting."],

["median_speech_text-realization", ">", "q3",
"The speech reflects realization or insight."],

["median_speech_text-excitement", ">", "q3",
"The spoken content reflects excitement or enthusiasm."],

["median_speech_text-surprise", ">", "q3",
"The speech expresses surprise."],

["median_speech_text-neutral", ">", "q3",
"The spoken content is emotionally neutral."],

]



### r-PPG

In [ ]:

heart_rate_criteria_list = [

# --------------------------------------------------
# Heart Rate Level
# Higher heart rate often reflects increased autonomic
# activation. Inspiration may involve motivational
# arousal, whereas sympathy may occur with calmer
# physiological regulation.
# --------------------------------------------------

["median_r_ppg-HR_Median", ">", "q3",
"The heart rate is relatively high, suggesting increased physiological activation. Elevated heart rate may occur when a person experiences emotionally engaging or motivational states such as inspiration."],

# ["median_r_ppg-HR_Median", "<=", "q1",
# "The heart rate is relatively low, suggesting a calmer physiological state. Lower heart rate may occur when emotional responses are reflective or empathic rather than highly arousing."],


# --------------------------------------------------
# rMSSD
# rMSSD reflects parasympathetic regulation and
# emotional flexibility. High rMSSD often appears
# during socially affiliative emotions such as
# compassion or sympathy.
# --------------------------------------------------

["median_r_ppg-rMSSD", ">", "q3",
"High rMSSD indicates strong parasympathetic activity and flexible heart rate regulation. This pattern is often observed during affiliative emotional states such as empathy or sympathy."],

["median_r_ppg-rMSSD", "<=", "q1",
"Low rMSSD indicates reduced parasympathetic regulation and lower heart rate variability, which may occur during emotionally arousing or activating experiences."],


# --------------------------------------------------
# pNN50
# Beat-to-beat variability indicator of parasympathetic
# control and emotional regulation.
# --------------------------------------------------

["median_r_ppg-pNN50", ">", "q3",
"A high pNN50 value indicates strong beat-to-beat variability and parasympathetic influence, which may occur during calm but emotionally engaged states such as empathic concern."],

["median_r_ppg-pNN50", "<=", "q1",
"A low pNN50 value suggests reduced beat-to-beat variability, which may occur when physiological arousal increases."],


# --------------------------------------------------
# SDNN
# Global HRV measure reflecting overall autonomic
# flexibility and emotional regulation capacity.
# --------------------------------------------------

["median_r_ppg-SDNN", ">", "q3",
"A higher SDNN indicates greater overall heart rate variability, suggesting flexible autonomic regulation that may accompany emotionally engaged but regulated responses."],

["median_r_ppg-SDNN", "<=", "q1",
"A lower SDNN indicates reduced overall heart rate variability, which may occur during emotionally intense or physiologically activated states."],


# --------------------------------------------------
# High Frequency (HF)
# HF reflects parasympathetic activity related to
# calm engagement and social emotional regulation.
# --------------------------------------------------

["median_r_ppg-HF", ">", "q3",
"High HF power suggests stronger parasympathetic nervous system activity. This pattern often appears during calm yet emotionally engaged states such as sympathy or compassionate concern."],

["median_r_ppg-HF", "<=", "q1",
"Low HF power suggests reduced parasympathetic influence and may indicate greater physiological activation."],


# --------------------------------------------------
# Low Frequency (LF)
# LF reflects mixed sympathetic and parasympathetic
# activity and may increase during emotional
# engagement or motivational states.
# --------------------------------------------------

["median_r_ppg-LF", ">", "q3",
"High LF power may reflect increased autonomic nervous system engagement, which can occur during emotionally meaningful or motivational experiences."],

# ["median_r_ppg-LF", "<=", "q1",
# "Low LF power suggests relatively lower autonomic variability."],


# --------------------------------------------------
# LF/HF ratio
# Indicates sympathetic–parasympathetic balance.
# Higher values often reflect sympathetic dominance.
# --------------------------------------------------

["median_r_ppg-LFHF", ">", "q3",
"A high LF/HF ratio may indicate stronger sympathetic influence, which can occur during motivational or activating emotional states such as inspiration."],

["median_r_ppg-LFHF", "<=", "q1",
"A low LF/HF ratio suggests stronger parasympathetic influence, which may occur during calm affiliative emotions such as sympathy."],

]

#### 2026/7/21 version

In [ ]:

# heart_rate_criteria_list = [

# # --------------------------------------------------
# # Heart Rate Level
# # Higher heart rate often reflects increased autonomic
# # activation. Inspiration may involve motivational
# # arousal, whereas sympathy may occur with calmer
# # physiological regulation.
# # --------------------------------------------------

# ["median_r_ppg-HR_Median", ">", "q3",
# "The heart rate is relatively high, suggesting increased physiological activation. Elevated heart rate may occur when a person experiences emotionally engaging or motivational states such as inspiration."],

# # ["median_r_ppg-HR_Median", "<=", "q1",
# # "The heart rate is relatively low, suggesting a calmer physiological state. Lower heart rate may occur when emotional responses are reflective or empathic rather than highly arousing."],


# # --------------------------------------------------
# # rMSSD
# # rMSSD reflects parasympathetic regulation and
# # emotional flexibility. High rMSSD often appears
# # during socially affiliative emotions such as
# # compassion or sympathy.
# # --------------------------------------------------

# ["median_r_ppg-rMSSD", ">", "q3",
# "High rMSSD indicates strong parasympathetic activity and flexible heart rate regulation. This pattern is often observed during affiliative emotional states such as empathy or sympathy."],

# # ["median_r_ppg-rMSSD", "<=", "q1",
# # "Low rMSSD indicates reduced parasympathetic regulation and lower heart rate variability, which may occur during emotionally arousing or activating experiences."],


# # --------------------------------------------------
# # pNN50
# # Beat-to-beat variability indicator of parasympathetic
# # control and emotional regulation.
# # --------------------------------------------------

# ["median_r_ppg-pNN50", ">", "q3",
# "A high pNN50 value indicates strong beat-to-beat variability and parasympathetic influence, which may occur during calm but emotionally engaged states such as empathic concern."],

# # ["median_r_ppg-pNN50", "<=", "q1",
# # "A low pNN50 value suggests reduced beat-to-beat variability, which may occur when physiological arousal increases."],


# # --------------------------------------------------
# # SDNN
# # Global HRV measure reflecting overall autonomic
# # flexibility and emotional regulation capacity.
# # --------------------------------------------------

# ["median_r_ppg-SDNN", ">", "q3",
# "A higher SDNN indicates greater overall heart rate variability, suggesting flexible autonomic regulation that may accompany emotionally engaged but regulated responses."],

# # ["median_r_ppg-SDNN", "<=", "q1",
# # "A lower SDNN indicates reduced overall heart rate variability, which may occur during emotionally intense or physiologically activated states."],


# # --------------------------------------------------
# # High Frequency (HF)
# # HF reflects parasympathetic activity related to
# # calm engagement and social emotional regulation.
# # --------------------------------------------------

# ["median_r_ppg-HF", ">", "q3",
# "High HF power suggests stronger parasympathetic nervous system activity. This pattern often appears during calm yet emotionally engaged states such as sympathy or compassionate concern."],

# # ["median_r_ppg-HF", "<=", "q1",
# # "Low HF power suggests reduced parasympathetic influence and may indicate greater physiological activation."],


# # --------------------------------------------------
# # Low Frequency (LF)
# # LF reflects mixed sympathetic and parasympathetic
# # activity and may increase during emotional
# # engagement or motivational states.
# # --------------------------------------------------

# ["median_r_ppg-LF", ">", "q3",
# "High LF power may reflect increased autonomic nervous system engagement, which can occur during emotionally meaningful or motivational experiences."],

# # ["median_r_ppg-LF", "<=", "q1",
# # "Low LF power suggests relatively lower autonomic variability."],


# # --------------------------------------------------
# # LF/HF ratio
# # Indicates sympathetic–parasympathetic balance.
# # Higher values often reflect sympathetic dominance.
# # --------------------------------------------------

# ["median_r_ppg-LFHF", ">", "q3",
# "A high LF/HF ratio may indicate stronger sympathetic influence, which can occur during motivational or activating emotional states such as inspiration."],

# # ["median_r_ppg-LFHF", "<=", "q1",
# # "A low LF/HF ratio suggests stronger parasympathetic influence, which may occur during calm affiliative emotions such as sympathy."],

# ]

### Thermal

In [ ]:

thermal_criteria_list = []

# --------------------------------------------
# Region definitions based on your prompt
# --------------------------------------------

FOREHEAD = {20,21,22,23,24}
NOSE = {27,28,29}
LEFT_CHEEK = {37,38,39,40,41}
RIGHT_CHEEK = {44,45,46}
MOUTH = {50,51,52,53}

# --------------------------------------------
# Helper function to interpret regions
# --------------------------------------------

def region_description(idx):

    if idx in FOREHEAD:
        return ("This landmark lies in the forehead region. Increased temperature in the forehead "
                "is often associated with heightened cognitive engagement and sympathetic activation. "
                "Such patterns may appear when a person experiences motivational emotions such as inspiration.")

    if idx in NOSE:
        return ("This landmark lies in the nasal region. Cooling in the nose area often occurs due to "
                "sympathetic vasoconstriction during emotional arousal. Nasal cooling has been observed "
                "during emotionally powerful or inspiring experiences.")

    if idx in LEFT_CHEEK or idx in RIGHT_CHEEK:
        return ("This landmark lies in the cheek region. Increased cheek temperature may reflect increased "
                "blood flow during affiliative emotions such as empathy or sympathy.")

    if idx in MOUTH:
        return ("This landmark lies near the mouth region. Temperature changes here may reflect breathing "
                "changes or expressive speech during emotional engagement.")

    return ("This landmark lies on the general facial surface. Temperature changes may reflect overall "
            "emotional engagement or autonomic activation.")


# --------------------------------------------
# Generate rules for all landmarks
# --------------------------------------------

# FOREHEAD = {20,21,22,23,24}
# NOSE = {27,28,29}
# LEFT_CHEEK = {37,38,39,40,41}
# RIGHT_CHEEK = {44,45,46}
# MOUTH = {50,51,52,53}

landmark_to_check = FOREHEAD.union(NOSE).union(LEFT_CHEEK).union(RIGHT_CHEEK).union(MOUTH)

for i in range(61):

    desc = region_description(i)

    # # mean temperature
    # thermal_criteria_list.append([
    #     f"thermal-mean_thermal_feature_{i}",
    #     ">",
    #     "q3",
    #     f"The average temperature at facial landmark {i} is relatively high. {desc}"
    # ])

    # # variance
    # thermal_criteria_list.append([
    #     f"thermal-variance_thermal_feature_{i}",
    #     ">",
    #     "q3",
    #     f"The temperature at facial landmark {i} varies substantially during the recording, "
    #     f"suggesting dynamic physiological responses and emotional engagement. {desc}"
    # ])

    # # max
    # thermal_criteria_list.append([
    #     f"thermal-max_thermal_feature_{i}",
    #     ">",
    #     "q3",
    #     f"The maximum temperature at facial landmark {i} reaches a high level during the recording. {desc}"
    # ])

    # end-to-begin change
    thermal_criteria_list.append([
        f"thermal-end_to_begin_thermal_feature_{i}",
        ">",
        "q3",
        f"The temperature at facial landmark {i} increases from the beginning to the end of the recording. {desc}"
    ])

    thermal_criteria_list.append([
        f"thermal-end_to_begin_thermal_feature_{i}",
        "<=",
        "q1",
        f"The temperature at facial landmark {i} decreases from the beginning to the end of the recording. {desc}"
    ])

#### 2026/7/21 Version

In [ ]:

# thermal_criteria_list = []

# # --------------------------------------------
# # Region definitions based on your prompt
# # --------------------------------------------

# FOREHEAD = {20,21,22,23,24}
# NOSE = {27,28,29}
# LEFT_CHEEK = {37,38,39,40,41}
# RIGHT_CHEEK = {44,45,46}
# MOUTH = {50,51,52,53}

# # --------------------------------------------
# # Helper function to interpret regions
# # --------------------------------------------

# # def region_description(idx):

# #     if idx in FOREHEAD:
# #         return ("This landmark lies in the forehead region. Increased temperature in the forehead "
# #                 "is often associated with heightened cognitive engagement and sympathetic activation. "
# #                 "Such patterns may appear when a person experiences motivational emotions such as inspiration.")

# #     if idx in NOSE:
# #         return ("This landmark is located in the nasal region. Emotional arousal can activate sympathetic vasoconstriction, directly reducing nasal temperature. Such temperature decreases have been observed during emotionally powerful or inspiring experiences..")

# #     if idx in LEFT_CHEEK or idx in RIGHT_CHEEK:
# #         return ("This landmark lies in the cheek region. Increased cheek temperature may reflect increased "
# #                 "blood flow during affiliative emotions such as empathy or sympathy.")

# #     if idx in MOUTH:
# #         return ("This landmark lies near the mouth region. Temperature changes here may reflect breathing "
# #                 "changes or expressive speech during emotional engagement.")

# #     return ("This landmark lies on the general facial surface. Temperature changes may reflect overall "
# #             "emotional engagement or autonomic activation.")

# def region_description(idx):
#     if idx in FOREHEAD:
#         return "Forehead temperature may increase with cognitive arousal during inspiration."

#     if idx in NOSE:
#         return "Nasal temperature may decrease through vasoconstriction during inspiration."

#     if idx in LEFT_CHEEK or idx in RIGHT_CHEEK:
#         return "Cheek temperature may increase with greater blood flow during sympathy."

#     if idx in MOUTH:
#         return "Mouth-area temperature may decrease with increased breathing during inspiration or sympathy."

#     return "Facial temperature may increase with autonomic activation during inspiration or sympathy."
    
# # --------------------------------------------
# # Generate rules for all landmarks
# # --------------------------------------------

# # FOREHEAD = {20,21,22,23,24}
# # NOSE = {27,28,29}
# # LEFT_CHEEK = {37,38,39,40,41}
# # RIGHT_CHEEK = {44,45,46}
# # MOUTH = {50,51,52,53}

# landmark_to_check = FOREHEAD.union(NOSE).union(LEFT_CHEEK).union(RIGHT_CHEEK).union(MOUTH)

# # for i in range(61):
# for i in landmark_to_check: 

#     desc = region_description(i)

#     # # mean temperature
#     # thermal_criteria_list.append([
#     #     f"thermal-mean_thermal_feature_{i}",
#     #     ">",
#     #     "q3",
#     #     f"The average temperature at facial landmark {i} is relatively high. {desc}"
#     # ])

#     # # variance
#     # thermal_criteria_list.append([
#     #     f"thermal-variance_thermal_feature_{i}",
#     #     ">",
#     #     "q3",
#     #     f"The temperature at facial landmark {i} varies substantially during the recording, "
#     #     f"suggesting dynamic physiological responses and emotional engagement. {desc}"
#     # ])

#     # # max
#     # thermal_criteria_list.append([
#     #     f"thermal-max_thermal_feature_{i}",
#     #     ">",
#     #     "q3",
#     #     f"The maximum temperature at facial landmark {i} reaches a high level during the recording. {desc}"
#     # ])

#     # end-to-begin change
#     thermal_criteria_list.append([
#         f"thermal-end_to_begin_thermal_feature_{i}",
#         ">",
#         "q3",
#         f"The temperature of this landmark increases - {desc}"
#     ])

#     thermal_criteria_list.append([
#         f"thermal-end_to_begin_thermal_feature_{i}",
#         "<=",
#         "q1",
#         f"The temperature of this landmark decreases - {desc}"
#     ])

### Facial

In [ ]:
facial_expression_criteria_list = []

# --------------------------------------------------
# Psychological interpretation of AUs
# --------------------------------------------------

def au_description(au):

    if au in ["AU06","AU12"]:
        return ("This Action Unit is part of the Duchenne smile pattern. "
                "The combination of AU06 (cheek raiser) and AU12 (lip corner puller) "
                "often indicates genuine positive emotion and may appear when a person "
                "experiences uplifting feelings such as admiration or inspiration.")

    if au in ["AU01","AU02"]:
        return ("This Action Unit raises the eyebrows. Eyebrow raising may occur during "
                "attention, curiosity, or emotional elevation, which sometimes appears "
                "when a person witnesses inspiring or meaningful events.")

    if au in ["AU01","AU04","AU15"]:
        return ("This Action Unit is associated with sadness-related expressions. "
                "Sadness cues may appear when a person experiences empathic concern "
                "or sympathy toward others.")

    if au in ["AU05","AU07"]:
        return ("This Action Unit relates to eye opening or tension around the eyes. "
                "These cues often indicate heightened attention or emotional engagement.")

    if au in ["AU25","AU26"]:
        return ("This Action Unit opens the mouth during speech or emotional reactions. "
                "Such movements may occur during surprise, admiration, or emotionally "
                "engaging experiences.")

    if au in ["AU09","AU10"]:
        return ("This Action Unit reflects nose wrinkling or upper lip raising, "
                "often associated with disgust reactions.")

    if au in ["AU23","AU24"]:
        return ("This Action Unit tightens the lips and may indicate emotional restraint "
                "or controlled emotional expression.")

    if au == "AU43":
        return ("This Action Unit reflects eye closure or blinking, which may occur "
                "during emotional emphasis or disengagement.")

    return ("This Action Unit reflects a facial muscle movement that may contribute "
            "to emotional expression depending on context.")


# --------------------------------------------------
# Emotion classifier signals
# --------------------------------------------------

emotion_labels = [
"anger","disgust","fear","happiness","sadness","surprise","neutral"
]

# --------------------------------------------------
# Action Units
# --------------------------------------------------

aus = [
"AU01","AU02","AU04","AU05","AU06","AU07","AU09","AU10","AU11","AU12",
"AU14","AU15","AU17","AU20","AU23","AU24","AU25","AU26","AU28","AU43"
]

# --------------------------------------------------
# Mean facial emotion predictions
# --------------------------------------------------

for emo in emotion_labels:

    facial_expression_criteria_list.append([
        f"median_{emo}",
        ">",
        "q3",
        f"The median probability of {emo} expression is relatively high, suggesting that the participant frequently displays facial patterns associated with {emo}."
    ])

    facial_expression_criteria_list.append([
        f"variance_{emo}",
        ">",
        "q3",
        f"The expression of {emo} varies considerably across time, suggesting dynamic emotional reactions."
    ])

    facial_expression_criteria_list.append([
        f"max_{emo}",
        ">",
        "q3",
        f"The participant briefly displays a strong facial expression associated with {emo}."
    ])

    facial_expression_criteria_list.append([
        f"end_to_begin_{emo}",
        ">",
        "q3",
        f"The facial expression of {emo} becomes stronger toward the end of the recording."
    ])


# --------------------------------------------------
# Action Unit criteria
# --------------------------------------------------

for au in aus:

    desc = au_description(au)

    facial_expression_criteria_list.append([
        f"median_{au}",
        ">",
        "q3",
        f"The average activation of {au} is relatively high. {desc}"
    ])

    facial_expression_criteria_list.append([
        f"variance_{au}",
        ">",
        "q3",
        f"The activation of {au} fluctuates substantially across the recording, "
        f"suggesting expressive facial movements. {desc}"
    ])

    facial_expression_criteria_list.append([
        f"max_{au}",
        ">",
        "q3",
        f"The maximum activation of {au} reaches a high level during the recording. {desc}"
    ])

    facial_expression_criteria_list.append([
        f"end_to_begin_{au}",
        ">",
        "q3",
        f"The activation of {au} increases from the beginning to the end of the recording. {desc}"
    ])

    facial_expression_criteria_list.append([
        f"end_to_begin_{au}",
        "<=",
        "q1",
        f"The activation of {au} decreases from the beginning to the end of the recording. {desc}"
    ])

## Tools

In [ ]:
if os.path.exists(path_pickle_df_list) and os.path.exists(path_pickle_df_names):
    try:
        all_df = data_loader.load_from_cache(path_pickle_df_list)
        all_df_name = data_loader.load_from_cache(path_pickle_df_names)
        print("Pickle files loaded successfully.")
    except Exception as e:
        print(f"Failed to load pickle files: {e}. Re-running processing.")
        all_df = None
        all_df_name = None
else:
    df_pose = data_loader.get_pose_features_without_landmarks_data_frame(record_path= path_pose, prefix = "pose")
    df_acoustics = data_loader.get_acoustics_df(path_acoustic_target, path_acoustic_baseline, prefix = "acoustics")
    df_speech_text = data_loader.get_speech_text_df(path_speech_text, prefix = "speech_text")
    df_facial = data_loader.get_facial_df(file_target_path = path_facial_target, file_baseline_path = path_facial_baseline, outlier_multiplier= outlier_multiplier, prefix = "facial")
    df_eye_movement = data_loader.get_eye_movement_df(path_eye_movement, prefix = "eye_move")
    df_remote_ppg = data_loader.get_r_ppg_df(record_path= path_r_ppg, df_info = df_pose[df_pose.columns[0:2]], prefix = "r_ppg")
    df_thermal = data_loader.get_thermal_df(path_thermal, prefix = "thermal")
    df_mm_llm = data_loader.get_mm_llm_df(path_mm_llm, prefix = "mm_llm")
    all_df = [df_acoustics,df_speech_text,df_facial,df_pose,df_eye_movement,df_remote_ppg,df_thermal,df_mm_llm]
    all_df_name = ["df_acoustics","df_speech_text","df_facial","df_pose","df_eye_movement","df_remote_ppg","df_thermal","df_mm_llm"]
    data_loader.save_to_cache(all_df, path_pickle_df_list)
    data_loader.save_to_cache(all_df_name, path_pickle_df_names)

HeartRateInterpreter = interpreter_builder.build_interpreter_class(heart_rate_criteria_list, class_name="HeartRateInterpreter", df= all_df[5])
heart_rate_interpreter = HeartRateInterpreter()

AcousticInterpreter = interpreter_builder.build_interpreter_class(acoustic_criteria_list, class_name="AcousticInterpreter", df= all_df[0])
acoustic_interpreter = AcousticInterpreter()

EyeMovementInterpreter = interpreter_builder.build_interpreter_class(eye_movement_criteria_list, class_name="EyeMovementInterpreter", df= all_df[4])
eye_movement_interpreter = EyeMovementInterpreter()

ThermalInterpreter = interpreter_builder.build_interpreter_class(thermal_criteria_list, class_name="ThermalInterpreter", df= all_df[6])
thermal_interpreter = ThermalInterpreter()

HeadPositionInterpreter = interpreter_builder.build_interpreter_class(head_position_criteria_list, class_name="HeadPositionInterpreter", df= all_df[3])
head_position_interpreter = HeadPositionInterpreter()

FacialExpressionInterpreter = interpreter_builder.build_interpreter_class(facial_expression_criteria_list, class_name="FacialExpressionInterpreter", df= all_df[2])
facial_expression_interpreter = FacialExpressionInterpreter()

TextEmotionInterpreter = interpreter_builder.build_interpreter_class(text_emotion_criteria_list, class_name="TextEmotionInterpreter", df= all_df[1])
text_emotion_interpreter = TextEmotionInterpreter()


In [ ]:

# interpreters_23 = [
#     thermal_interpreter,
#     head_position_interpreter,
#     facial_expression_interpreter,
#     heart_rate_interpreter
    # facial_expression_interpreter,
    # head_position_interpreter,
    # eye_movement_interpreter,
    # acoustic_interpreter,
    # text_emotion_interpreter
# ]

interpreters_25 = [
    thermal_interpreter,
    heart_rate_interpreter
]

In [ ]:
# tools_template_23 = """
#     The following features were extracted using external analytical tools. Use these results as supporting evidence in your assessment:

#     Thermal imaging analysis result: {0}
#     Head movement analysis result: {1}
#     Facial Action Unit (AU) analysis result: {2}
#     Heart rate variability analysis result: {3}

#     Be especially careful when identifying the emotion of inspiration, as it may be less obvious than other emotional states.
# """

tools_template_25="""

Sympathy is the emotional response of the primary participant when perceiving that another person is in need of help or support. Inspiration is the emotional response of the primary participant when observing behaviors that are admirable, such as performing exceptional actions or helping others.

The following features were obtained using external analytical tools. Use these results as supporting evidence in your assessment when determining the scores for sympathy and inspiration.

Thermal imaging analysis result: {0}

Heart rate variability analysis result: {1}

"""

# tools_template_25="""
#     The following features were obtained using external analytical tools. Use these results as supporting evidence in your assessment.

#     Eye-tracking analysis result: {0}
#     Audio analysis result: {1}
#     Linguistic emotion analysis result: {2}
# """

In [ ]:
experiment_helper_4 = experiment_helper.copy()
experiment_helper_4.define_experiment(
    participant_sn_range= list_users,
    path_pickle_df_video= os.path.join(path_paper, "temp", "pickle", "merged_video_25.plk"),
    path_pickle_df_experiment_results=os.path.join(path_paper, "temp", "pickle", "df_experiment_4.plk"),
    path_compressed_video= os.path.join(path_paper, "temp", "compressed_video"),
    experiment_step=25,
    video_only=False,
    interpreters = interpreters_25,
    template_main=template_main_25,
    template_cot = "",
    template_knowledge_injection= "",
    template_tools= tools_template_25,
    attach_video = True
)


## Generate Predictions

In [ ]:
run_experiments = True
if run_experiments:
    # num_folds = 5
    df_result_25_4 = experiment_helper_4.run_exp_moral_elevation(over_ride_existing=False, max_workers=3)
else:
    df_result_25_4 = experiment_helper_4.get_df_experiment_results()
run_experiments = False

In [ ]:
df_result_25_4 = experiment_helper_4.get_df_experiment_results()

In [ ]:
df_result_25_4

In [ ]:
df_result_25_4.to_csv("df_result_25_4.csv")

## Print Out Results

In [ ]:
import pandas as pd
import numpy as np

# =========================
# Function: update predictions based on median ± K%
# =========================
def update_by_median_k_percent(
    df_result_25_2: pd.DataFrame,
    df_result_25_7: pd.DataFrame,
    k_percent: float = 0.1  # For example, 0.1 = 10%
):
    """
    In df_result_25_2, replace the sympathy / inspiration values that fall within the median ± K% range,
    with the corresponding values for the same Participant SN in df_result_25_7

    Parameters
    ----------
    df_result_25_2 : original results
    df_result_25_7 : replacement results
    k_percent : percentage range (e.g., 0.1 = ±10%)

    Returns
    -------
    df_result_25_updated : updated DataFrame
    """

    # Copy to avoid modifying original data
    df_updated = df_result_25_2.copy()

    # merge corresponding values from another dataframe
    df_merged = df_updated.merge(
        df_result_25_7[["Participant SN", "sympathy", "inspiration"]],
        on="Participant SN",
        how="left",
        suffixes=("", "_new")
    )

    # ====== Processing sympathy ======
    median_sym = df_merged["sympathy"].median()
    lower_sym = median_sym * (1 - k_percent)
    upper_sym = median_sym * (1 + k_percent)

    mask_sym = df_merged["sympathy"].between(lower_sym, upper_sym)

    # Replace
    df_merged.loc[mask_sym, "sympathy"] = df_merged.loc[mask_sym, "sympathy_new"]

    # ====== Processing inspiration ======
    median_ins = df_merged["inspiration"].median()
    lower_ins = median_ins * (1 - k_percent)
    upper_ins = median_ins * (1 + k_percent)

    mask_ins = df_merged["inspiration"].between(lower_ins, upper_ins)

    # Replace
    df_merged.loc[mask_ins, "inspiration"] = df_merged.loc[mask_ins, "inspiration_new"]

    # ====== Cleanup ======
    df_result_25_updated = df_merged.drop(columns=["sympathy_new", "inspiration_new"])

    # ====== Debug print ======
    print(f"Sympathy replaced rows: {mask_sym.sum()}")
    print(f"Inspiration replaced rows: {mask_ins.sum()}")

    return df_result_25_updated


# =========================
# Usage
# =========================
df_result_25_4_updated = update_by_median_k_percent(
    df_result_25_3,
    df_result_25_4,
    k_percent=20  # For example ±10%
)

df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_25_4_updated],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Exp. 8 V+A+T+ P",
    font_size = 20
)


In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_25_4],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Exp 4. V+A+T+P",
    font_size = 20
)

In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_25_4],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Exp 4. V+A+T+P",
    font_size = 20
)

# Experiment 14 - Video + Audio  + Physio (manuscript Exp 16, supplementary control; Appendix D.1)

## Definition

In [ ]:
part_first_no_speech_text_14 = """
The first attached video is a behavioral recording captured during an experiment in which an individual watches a short film.

The individual at the center of the frame is the primary participant and the only subject to be analyzed.

You must make a clear and definitive judgment on two emotional dimensions.

IMPORTANT: You should use all available visual, heart-rate, thermal, and acoustic information. However, you must not analyze, infer, or use the semantic content of the participant's speech under any circumstances.

This constraint is strict and non-negotiable.
"""

template_no_speech_text_14 = part_first_no_speech_text_14 + part_main + last_part

In [ ]:
experiment_helper_14 = experiment_helper.copy()
experiment_helper_14.define_experiment(
    participant_sn_range= list_users,
    path_pickle_df_video= os.path.join(path_paper, "temp", "pickle", "merged_video_25.plk"),
    path_pickle_df_experiment_results=os.path.join(path_paper, "temp", "pickle", "df_experiment_14.plk"),
    path_compressed_video= os.path.join(path_paper, "temp", "compressed_video"),
    experiment_step=25,
    video_only=False,
    interpreters = interpreters_25,
    template_main=template_no_speech_text_14, #template_main_25,
    template_cot = "",
    template_knowledge_injection= "",
    template_tools= tools_template_25,
    attach_video = True
)


## Generate Predictions

In [ ]:
run_experiments = True
if run_experiments:
    df_result_25_14 = experiment_helper_14.run_exp_moral_elevation(over_ride_existing=False, max_workers=3)
    df_result_25_14.to_csv("experiment_14.csv")
else:
    df_result_25_14 = experiment_helper_14.get_df_experiment_results()
    df_result_25_14.to_csv("experiment_14.csv")

run_experiments = False


## Print Out Results

In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_25_14],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Exp 14. V+A+P",
    font_size = 20
)

# Experiment 9 - Agentic

## Definition

In [ ]:
import pickle
import os
import numpy as np
import re

DEBUG = False
MAX_ITER = 1
N_SAMPLES = 3
list_users = list(range(1, 120))
# ===== DataFrame pickle path =====
pickle_path_df = os.path.join(path_paper, "temp", "pickle", "df_experiment_9.plk")

### extract_analysis

In [ ]:
import re, json
import pandas as pd

def extract_analysis(llm_result):
    # === Correction: handle pandas Series ===
    if isinstance(llm_result, pd.Series):
        if len(llm_result) == 0:
            return _default_analysis("empty_series")
        llm_result = llm_result.iloc[0]

    # === Idiot-proof: None or non-string ===
    if not isinstance(llm_result, str):
        return _default_analysis("not_string")

    try:
        match = re.search(r"\[ANALYSIS_JSON\](.*?)\[/ANALYSIS_JSON\]", llm_result, re.S)
        if match:
            return json.loads(match.group(1))
    except Exception as e:
        print("Parse error:", e)

    return _default_analysis("parse_failed")


def _default_analysis(reason):
    return {
        "sympathy": None,
        "inspiration": None,
        "confidence": 0.5,
        "reliability": 0.5,
        "uncertainty": {"level": 0.5, "reason": reason},
        "evidence": [],
        "temporal_pattern": "",
        "limitations": [reason]
    }

### templates

In [ ]:
part_first_llm_v_a = """
The first attached video is a behavioral recording captured during an experimental procedure in which a person is watching a short film.

The individual located at the center of the frame is the primary participant and is the only subject to be analyzed.

Sympathy is the emotional response of the primary participant when perceiving that another person is in need of help or support. Inspiration is the emotional response of the primary participant when observing behaviors that are admirable, such as performing exceptional actions or helping others.

You must provide a final judgment on two emotional dimensions, while explicitly assessing confidence, reliability, and uncertainty.
"""

part_first_thermal = """

Sympathy is the emotional response of the primary participant when perceiving that another person is in need of help or support. Inspiration is the emotional response of the primary participant when observing behaviors that are admirable, such as performing exceptional actions or helping others.

The following features were obtained from external facial thermal analysis. Use these results as supporting evidence in your assessment when determining the scores for sympathy and inspiration.
"""

part_first_remote_ppg = """

Sympathy is the emotional response of the primary participant when perceiving that another person is in need of help or support. Inspiration is the emotional response of the primary participant when observing behaviors that are admirable, such as performing exceptional actions or helping others.

The following features were obtained from external remote photoplethysmography (rPPG) analysis. Use these results as supporting evidence in your assessment when determining the scores for sympathy and inspiration.

"""


template_agent_response = """
[ANALYSIS_JSON]
{
"sympathy": X,
"inspiration": Y,
"confidence": C,
"reliability": R,
"explanation": Z
"uncertainty": {
    "level": U,
    "reason": ""
},
"evidence": [],
"temporal_pattern": "",
"limitations": []
}
[/ANALYSIS_JSON]
"""

In [ ]:
template_llm_v_a = part_first_llm_v_a + part_main + template_agent_response +  last_part
template_thermal = part_first_thermal + part_main + template_agent_response +  last_part
template_remote_ppg = part_first_remote_ppg + part_main + template_agent_response +  last_part

interpreter_thermal = [
    thermal_interpreter,
]
interpreter_remote_ppg = [
    heart_rate_interpreter
]

tools_template_thermal="""
    The following features were obtained using external analytical tools. Use these results as supporting evidence in your assessment.
    Thermal imaging analysis result: {0}
"""

tools_template_remote_ppg="""
    The following features were obtained using external analytical tools. Use these results as supporting evidence in your assessment.
    Heart rate variability analysis result: {0}
"""

### generate_multiple

In [ ]:
def generate_multiple(experiment_helper_input, feedback_memory, n=3):
    
    results = []
    max_workers = 1

    # ===== Prompt Injection =====
    this_template_llm_v_a = template_llm_v_a
    this_template_thermal = template_thermal
    this_template_remote_ppg = template_remote_ppg

    if feedback_memory:
        this_template_llm_v_a = this_template_llm_v_a + "\n\n[REVISION]\n" + feedback_memory

    this_llm_client = experiment_helper_input.get_llm_client()
    this_llm_client.set_temperature(2) # gemini is 0~2
    experiment_helper_input.set_llm_client(this_llm_client)

    experiment_helper_llm_v_a = experiment_helper_input.copy()
    experiment_helper_llm_v_a.define_experiment(
        participant_sn_range=list_user,
        path_pickle_df_video=os.path.join(path_paper, "temp", "pickle", "merged_video_25.plk"),
        path_pickle_df_experiment_results=os.path.join(path_paper, "temp", "pickle", "df_experiment_9_temp.plk"),
        path_compressed_video=os.path.join(path_paper, "temp", "compressed_video"),
        experiment_step=25,
        video_only=False,
        interpreters=[],
        template_main=this_template_llm_v_a,
        template_cot="",
        template_knowledge_injection="",
        template_tools="",
        attach_video=True
    )


    experiment_helper_thermal = experiment_helper_input.copy()
    experiment_helper_thermal.define_experiment(
        participant_sn_range=list_user,
        path_pickle_df_video=os.path.join(path_paper, "temp", "pickle", "merged_video_25.plk"),
        path_pickle_df_experiment_results=os.path.join(path_paper, "temp", "pickle", "df_experiment_9_temp.plk"),
        path_compressed_video=os.path.join(path_paper, "temp", "compressed_video"),
        experiment_step=25,
        video_only=False,
        interpreters=interpreter_thermal,
        template_main=this_template_thermal,
        template_cot="",
        template_knowledge_injection="",
        template_tools=tools_template_thermal,
        attach_video=False
    )

    experiment_helper_remote_ppg = experiment_helper_input.copy()
    experiment_helper_remote_ppg.define_experiment(
        participant_sn_range=list_user,
        path_pickle_df_video=os.path.join(path_paper, "temp", "pickle", "merged_video_25.plk"),
        path_pickle_df_experiment_results=os.path.join(path_paper, "temp", "pickle", "df_experiment_9_temp.plk"),
        path_compressed_video=os.path.join(path_paper, "temp", "compressed_video"),
        experiment_step=25,
        video_only=False,
        interpreters=interpreter_remote_ppg,
        template_main=this_template_remote_ppg,
        template_cot="",
        template_knowledge_injection="",
        template_tools=tools_template_remote_ppg,
        attach_video=False
    )

    def generate_result(experiment_helper, results, modality):
        df = experiment_helper.run_exp_moral_elevation(
            over_ride_existing=True,
            max_workers=max_workers
        )

        # 1. df does not exist or is empty
        if df is None or len(df) == 0:
            return

        # 2. Field does not exist
        if "LLM_Result" not in df.columns:
            return

        # 3. Safe retrieval of first column
        row = df.iloc[0]

        # 4. LLM_Result is empty or NaN
        llm_result = row.get("LLM_Result")
        if llm_result is None or (isinstance(llm_result, float) and pd.isna(llm_result)):
            return

        # 5. Parse failure protection
        try:
            analysis = extract_analysis(llm_result)
        except Exception:
            return

        if not isinstance(analysis, dict):
            return

        # 6. Critical field validation
        sympathy = analysis.get("sympathy")
        if sympathy is None:
            return

        # 7. Incorporate modality and persist results
        analysis["modality"] = modality
        results.append(analysis)





    def chek_results(results_input):

        primary_modalities = {"visual and sound"}

        def safe_float(x, default=0.0):
            try:
                return float(x)
            except:
                return default
        # ===== 1. primary filtering =====
        # primary = [
        #     a for a in results_input
        #     if a.get("modality") in primary_modalities
        #     and a.get("confidence", 0) > 0.5
        # ]
        # ===== 1. primary filtering =====
        primary = [
            a for a in results_input
            if a.get("modality") in primary_modalities
            and safe_float(a.get("confidence", 0)) > 0.5
        ]

    def simple_fuse(candidates):
        if not candidates:
            return None

        def to_float(x, default=0.0):
            try:
                return float(x)
            except (TypeError, ValueError):
                return default

        # If uncertainty is a category (low/medium/high), perform mapping
        UNC_MAP = {
            "low": 0.1,
            "medium": 0.5,
            "high": 0.9
        }

        def parse_uncertainty(u):
            if isinstance(u, dict):
                u = u.get("level", 0.5)

            # numeric string
            try:
                return float(u)
            except:
                pass

            # Category
            if isinstance(u, str):
                return UNC_MAP.get(u.lower(), 0.5)

            return 0.5  # fallback

        # -------- Data extraction (all forced to numeric) --------
        sym = np.array([to_float(a.get("sympathy")) for a in candidates], dtype=float)
        ins = np.array([to_float(a.get("inspiration")) for a in candidates], dtype=float)
        conf = np.array([to_float(a.get("confidence"), 1.0) for a in candidates], dtype=float)
        rel = np.array([to_float(a.get("reliability"), 1.0) for a in candidates], dtype=float)
        unc = np.array([parse_uncertainty(a.get("uncertainty")) for a in candidates], dtype=float)

        # -------- Weight calculation (stable version) --------
        weights = conf * rel * (1.0 - unc)

        # Guard against NaN / Inf
        weights = np.nan_to_num(weights, nan=0.0, posinf=0.0, neginf=0.0)

        if weights.sum() <= 1e-8:
            weights = np.ones_like(weights)

        weights = weights / weights.sum()

        # -------- Fusion --------
        return {
            "sympathy": int(round(np.sum(sym * weights))),
            "inspiration": int(round(np.sum(ins * weights))),
            "confidence": float(np.sum(conf * weights)),
            "reliability": float(np.sum(rel * weights)),
            "uncertainty": float(np.sum(unc * weights)),
            "weights": weights.tolist(),   # Optional: Debug use
            "sources": candidates
        }
        # def simple_fuse(candidates):
        #     if not candidates:
        #         return None

        #     sym = np.array([a["sympathy"] for a in candidates])
        #     ins = np.array([a["inspiration"] for a in candidates])
        #     conf = np.array([a["confidence"] for a in candidates])
        #     rel = np.array([a["reliability"] for a in candidates])
        #     unc = np.array([a["uncertainty"]["level"] for a in candidates])

        #     weights = conf * rel * (1 - unc)
        #     if weights.sum() == 0:
        #         weights = np.ones_like(weights)
        #     weights = weights / weights.sum()

        #     return {
        #         "sympathy": int(round(np.sum(sym * weights))),
        #         "inspiration": int(round(np.sum(ins * weights))),
        #         "confidence": float(np.sum(conf * weights)),
        #         "reliability": float(np.sum(rel * weights)),
        #         "uncertainty": float(np.sum(unc * weights)),
        #         "sources": candidates
        #     }

        fused = simple_fuse(primary)
        return fused


    
    for _ in range(n):
        generate_result(experiment_helper_llm_v_a, results, "visual and sound")
    
    fused = None
    
    if len(results)>0:
        fused = chek_results(results)

    if fused is None or fused["confidence"] < 0.8:
        for _ in range(n):
            generate_result(experiment_helper_thermal, results, "thermal")
            generate_result(experiment_helper_remote_ppg, results, "remote ppg")
            

    return results

In [ ]:
import numpy as np

def fusion_agent(analyses, 
                 conf_th=0.6, 
                 rel_th=0.6, 
                 agreement_tol=2):

    def safe_float(x, default=0.0):
        try:
            return float(x)
        except:
            return default
    # ===== 1. Filter low quality =====
    # valid = [
    #     a for a in analyses
    #     if a.get("sympathy") is not None
    #     and a.get("confidence", 0) >= conf_th
    #     and a.get("reliability", 0) >= rel_th
    # ]

    valid = [
        a for a in analyses
        if a.get("sympathy") is not None
        and safe_float(a.get("confidence", 0)) >= conf_th
        and safe_float(a.get("reliability", 0)) >= rel_th
    ]

    # fallback (keep original logic)
    if not valid:
        return analyses[0]

    # ===== 2. Extract numeric values =====
    sym = np.array([a["sympathy"] for a in valid])
    ins = np.array([a["inspiration"] for a in valid])
    conf = np.array([a["confidence"] for a in valid])
    rel = np.array([a["reliability"] for a in valid])
    unc = np.array([a["uncertainty"]["level"] for a in valid])

    # ===== 3. Consistency Detection =====
    sym_std = np.std(sym)
    ins_std = np.std(ins)

    # ===== 4. Weights (Critical) =====
    weights = conf * rel * (1 - unc)

    if weights.sum() == 0:
        weights = np.ones_like(weights)

    weights = weights / weights.sum()

    # ===== 5. Weighted fusion =====
    fused_sym = int(round(np.sum(sym * weights)))
    fused_ins = int(round(np.sum(ins * weights)))

    fused_conf = float(np.sum(conf * weights))
    fused_rel = float(np.sum(rel * weights))

    # ===== 6. Inconsistency penalty (applied to uncertainty) =====
    disagreement_penalty = (sym_std + ins_std) / 10.0

    fused_unc_level = float(
        np.clip(
            np.sum(unc * weights) + disagreement_penalty,
            0, 1
        )
    )

    # ===== 7. Select representative samples (avoid arbitrary combinations) =====
    best_idx = np.argmax(weights)
    best = valid[best_idx]

    return {
        "sympathy": fused_sym,
        "inspiration": fused_ins,
        "confidence": fused_conf,
        "reliability": fused_rel,
        "uncertainty": {
            "level": fused_unc_level,
            "reason": best["uncertainty"].get("reason", "")
        },
        "evidence": best.get("evidence", []),
        "temporal_pattern": best.get("temporal_pattern", ""),
        "limitations": best.get("limitations", [])
    }

### result_synthesis_agent

In [ ]:
# import json
# import re
# import numpy as np

# def result_synthesis_agent(llm_client, analyses):

#     primary_modalities = {"visual and sound"}

#     # ===== 1. primary filtering =====
#     primary = [
#         a for a in analyses
#         if a.get("modality") in primary_modalities
#         and a.get("confidence", 0) > 0.5
#     ]

#     def simple_fuse(candidates):
#         if not candidates:
#             return None

#         sym = np.array([a["sympathy"] for a in candidates])
#         ins = np.array([a["inspiration"] for a in candidates])
#         conf = np.array([a["confidence"] for a in candidates])
#         rel = np.array([a["reliability"] for a in candidates])
#         unc = np.array([a["uncertainty"]["level"] for a in candidates])

#         weights = conf * rel * (1 - unc)
#         if weights.sum() == 0:
#             weights = np.ones_like(weights)
#         weights = weights / weights.sum()

#         return {
#             "sympathy": int(round(np.sum(sym * weights))),
#             "inspiration": int(round(np.sum(ins * weights))),
#             "confidence": float(np.sum(conf * weights)),
#             "reliability": float(np.sum(rel * weights)),
#             "uncertainty": float(np.sum(unc * weights)),
#             "sources": candidates
#         }

#     fused = simple_fuse(primary)

#     # ===== 2. fallback to secondary =====
#     if fused is None or fused["confidence"] < 0.8:
#         secondary = [
#             a for a in analyses
#             if a.get("modality") not in primary_modalities
#             and a.get("confidence", 0) > 0.5
#         ]
#         fused = simple_fuse((primary or []) + secondary)

#     if fused is None:
#         fused = analyses[0]

# ===== 3. LLM synthesis (key modification) =====
#     analysis_result = json.dumps(fused, indent=2)

#     prompt = f"""
# You are an expert in multimodal emotion recognition.

# Given the following candidate analyses (already filtered and fused):

# {analysis_result}

# Your task is to produce a FINAL decision.

# ----------------------------------
# Output MUST include:

# 1. Final sympathy score (1–10)
# 2. Final inspiration score (1–10)
# 3. A clear explanation combining the most reliable evidence
# 4. An overall confidence score (0.0–1.0)

# ----------------------------------
# Guidelines:

# - Prioritize visual and sound modalities
# - Only use high-confidence sources
# - Resolve conflicts between modalities
# - Do NOT simply average — reason about evidence
# - Confidence should reflect:
#   - agreement between sources
#   - strength of evidence
#   - consistency of signals

# ----------------------------------
# Output ONLY JSON:

# {{
# "sympathy": int,
# "inspiration": int,
# "explanation": "string",
# "confidence": float
# }}

# Rules:
# - Do NOT output extra text
# - explanation must be non-empty
# - confidence must NOT be 0.0 unless no evidence exists
# """

#     response = llm_client.chat(prompt, stream=False, verbose=True)

#     # ===== parsing =====
#     try:
#         return json.loads(response)
#     except:
#         pass

#     match = re.search(r"```(?:json)?\s*(.*?)\s*```", response, re.S)
#     if match:
#         try:
#             return json.loads(match.group(1))
#         except:
#             pass

#     match = re.search(r"\{.*\}", response, re.S)
#     if match:
#         try:
#             return json.loads(match.group(0))
#         except:
#             pass

# fallback (ensure structure consistency)
#     return {
#         "sympathy": fused.get("sympathy", 5),
#         "inspiration": fused.get("inspiration", 5),
#         "explanation": "Fallback synthesis due to parsing failure.",
#         "confidence": fused.get("confidence", 0.5)
#     }

In [ ]:
def result_synthesis_agent(llm_client, analyses):

    import numpy as np
    import json
    import re

    primary_modalities = {"visual and sound"}

    # ---------- Tool functions ----------
    def safe_float(x, default=0.0):
        try:
            return float(x)
        except (TypeError, ValueError):
            return default

    UNC_MAP = {
        "low": 0.1,
        "medium": 0.5,
        "high": 0.9
    }

    def parse_uncertainty(u):
        if isinstance(u, dict):
            u = u.get("level", 0.5)

        # numeric string
        try:
            return float(u)
        except:
            pass

        # Category
        if isinstance(u, str):
            return UNC_MAP.get(u.lower(), 0.5)

        return 0.5

    # ---------- Preprocessing (critical fix points) ----------
    for a in analyses:
        a["confidence"] = safe_float(a.get("confidence"), 0.0)
        a["reliability"] = safe_float(a.get("reliability"), 1.0)
        a["sympathy"] = safe_float(a.get("sympathy"), 5.0)
        a["inspiration"] = safe_float(a.get("inspiration"), 5.0)
        a["uncertainty"] = {
            "level": parse_uncertainty(a.get("uncertainty"))
        }

    # ===== 1. primary filtering =====
    primary = [
        a for a in analyses
        if a.get("modality") in primary_modalities
        and a["confidence"] > 0.5   # Already safe
    ]

    # ---------- fuse ----------
    def simple_fuse(candidates):
        if not candidates:
            return None

        sym = np.array([a["sympathy"] for a in candidates], dtype=float)
        ins = np.array([a["inspiration"] for a in candidates], dtype=float)
        conf = np.array([a["confidence"] for a in candidates], dtype=float)
        rel = np.array([a["reliability"] for a in candidates], dtype=float)
        unc = np.array([a["uncertainty"]["level"] for a in candidates], dtype=float)

        weights = conf * rel * (1.0 - unc)
        weights = np.nan_to_num(weights)

        if weights.sum() <= 1e-8:
            weights = np.ones_like(weights)

        weights = weights / weights.sum()

        return {
            "sympathy": int(round(np.sum(sym * weights))),
            "inspiration": int(round(np.sum(ins * weights))),
            "confidence": float(np.sum(conf * weights)),
            "reliability": float(np.sum(rel * weights)),
            "uncertainty": float(np.sum(unc * weights)),
            "sources": candidates
        }

    fused = simple_fuse(primary)

    # ===== 2. fallback to secondary =====
    if fused is None or fused["confidence"] < 0.8:
        try:
            secondary = [
                a for a in analyses
                if a.get("modality") not in primary_modalities
                and a["confidence"] > 0.5   # Already safe
            ]
            fused = simple_fuse((primary or []) + secondary)
        except Exception as e:
            fused = simple_fuse((primary))
        

    if fused is None:
        fused = analyses[0]

    # ===== 3. LLM synthesis =====
    analysis_result = json.dumps(fused, indent=2)

    prompt = f"""
You are an expert in multimodal emotion recognition.

Given the following candidate analyses (already filtered and fused):

{analysis_result}

Your task is to produce a FINAL decision.

----------------------------------
Output MUST include:

1. Final sympathy score (1–10)
2. Final inspiration score (1–10)
3. A clear explanation combining the most reliable evidence
4. An overall confidence score (0.0–1.0)

----------------------------------
Guidelines:

- Prioritize visual and sound modalities
- Only use high-confidence sources
- Resolve conflicts between modalities
- Do NOT simply average — reason about evidence
- Confidence should reflect:
  - agreement between sources
  - strength of evidence
  - consistency of signals

----------------------------------
Output ONLY JSON:

{{
"sympathy": int,
"inspiration": int,
"explanation": "string",
"confidence": float
}}
"""

    response = llm_client.chat(prompt, stream=False, verbose=True)

    # ===== parsing =====
    try:
        return json.loads(response)
    except:
        pass

    match = re.search(r"```(?:json)?\s*(.*?)\s*```", response, re.S)
    if match:
        try:
            return json.loads(match.group(1))
        except:
            pass

    match = re.search(r"\{.*\}", response, re.S)
    if match:
        try:
            return json.loads(match.group(0))
        except:
            pass

    return {
        "sympathy": fused.get("sympathy", 5),
        "inspiration": fused.get("inspiration", 5),
        "explanation": "Fallback synthesis due to parsing failure.",
        "confidence": fused.get("confidence", 0.5)
    }

In [ ]:
class AgentMemory:
    def __init__(self):
        self.history = []

    def add(self, analysis, critic):
        self.history.append({
            "analysis": analysis,
            "critic": critic
        })

    def summarize(self):
        return json.dumps(self.history[-2:], indent=2)

In [ ]:
def controller(critic_output, memory_summary):

    try:
        redo = False
        feedback = []

        if critic_output["consistency"] < 0.6:
            feedback.append("Improve consistency between reasoning and evidence")
            redo = True

        if "evidence" in str(critic_output["issues"]):
            feedback.append("Provide stronger observable evidence")
            redo = True

        if critic_output["target"] == "confidence":
            feedback.append("Re-evaluate confidence level")
            redo = True

        if memory_summary:
            feedback.append(f"Previous attempts:\n{memory_summary}")

        return {
            "action": "redo" if redo else "accept",
            "feedback": "\n".join(feedback)
        }

    except Exception as e:
        return {
            "action": "redo",
            "feedback": "Error"
        }

## Run Experiment

In [ ]:
# ===== Load or initialize df_video_23 =====
if os.path.exists(pickle_path_df):
    with open(pickle_path_df, "rb") as f:
        df_result_9 = pickle.load(f)
        df_result_9.to_csv("experiment_9.csv")
else:
    df_result_9 = df_video_25

In [ ]:



# ===== Update + save function =====
def update_df_and_save(df_video, result):

    participant = result["Participant"][0]

    # Type unification (critical)
    df_video["Participant SN"] = df_video["Participant SN"].astype(int)

    mask = df_video["Participant SN"] == int(participant)

    if not mask.any():
        print(f"[Warning] Participant {participant} not found.")
        return df_video

    # ===== Update =====
    df_video.loc[mask, "sympathy"] = result["sympathy"]
    df_video.loc[mask, "inspiration"] = result["inspiration"]
    df_video.loc[mask, "confidence"] = result["confidence"]
    df_video.loc[mask, "iterations"] = result["iterations"]
    df_video.loc[mask, "explanation"] = result["explanation"]
    df_video.loc[mask, "LLM_Result"] = result["raw"]

    # ===== Save =====
    with open(pickle_path_df, "wb") as f:
        pickle.dump(df_video, f)

    return df_video


# ===== User Completed =====
# done_users = set(
#     df_result_8.loc[df_result_8["sympathy"].notna(), "Participant SN"].astype(int)
# )
done_users = set(
    df_result_9.loc[
        df_result_9["sympathy"].notna() &
        (df_result_9["sympathy"].astype(str).str.strip() != ""),
        "Participant SN"
    ].astype(int)
)

print(f"Already done: {len(done_users)} users")


# Create a set of Participant SNs
participant_set = set(df_video_25["Participant SN"].dropna().astype(int))

# ===== Main loop =====
for user_id in list_users:

    if user_id in done_users:
        print(f"SKIP USER {user_id} (already processed)")
        continue

    # Skip if not in df_video
    if user_id not in participant_set:
        print(f"SKIP USER {user_id} (not in df_video)")
        continue

    list_user = [user_id]

    memory = AgentMemory()

    iteration = 0
    done = False
    feedback_memory = ""

    print(f"\n===== USER {list_user} =====")

    while not done and iteration < MAX_ITER:

        # ===== DEBUG mode =====
        if DEBUG:
            analyses = {
                "sympathy": 8,
                "inspiration": 1,
                "confidence": 0.8,
                "reliability": 0.7,
                "uncertainty": 0.3,
                "sources": []
            }
        else:
            analyses = generate_multiple(
                experiment_helper,
                feedback_memory,
                N_SAMPLES
            )

        # ===== Synthesis =====
        if not analyses:
            continue
        if len(analyses)<1:
            continue
        
        result_synthesis = result_synthesis_agent(llm_client, analyses)

        # ===== Memory =====
        if result_synthesis.get("explanation"):
            memory.add(analyses, result_synthesis["explanation"])

        # ===== Determining whether to accept =====
        if result_synthesis["confidence"] >= 0.7:

            done = True
            analyses_text = json.dumps(analyses, indent=2)
            result = {
                "Participant": list_user,
                "sympathy": result_synthesis["sympathy"],
                "inspiration": result_synthesis["inspiration"],
                "confidence": result_synthesis["confidence"],
                "iterations": iteration,
                "explanation": result_synthesis["explanation"],
                "raw": analyses_text
            }

            # ===== Updating DataFrame + Saving =====
            df_result_9 = update_df_and_save(df_result_9, result)

            done_users.add(user_id)

            print(f"ACCEPTED & SAVED ({user_id})")

        else:
            feedback_memory = result_synthesis["explanation"]
            iteration += 1

            if iteration < MAX_ITER:
                print("REDO:", feedback_memory)
            else:
                print(f"User: {user_id}, no result")

## Print Out Results

In [ ]:
# df_result_25_2[["sympathy","inspiration","Group Name","explanation"]]

with pd.option_context("display.max_colwidth", None):
    display(df_result_25_3[0:11][["Group Name","Participant SN", "sympathy","inspiration","LLM_Result"]])

In [ ]:
df_result_9 = pd.read_pickle(pickle_path_df)
# df_result_8[["sympathy","inspiration","Group Name","LLM_Result"]]

with pd.option_context("display.max_colwidth", None):
    #display(df_result_8[0:11][["Group Name","Participant SN", "sympathy","inspiration","explanation", "LLM_Result"]])
    display(df_result_9[["Group Name","Participant SN", "sympathy","inspiration","explanation", "LLM_Result"]])

In [ ]:
df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_9],
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Experiment 9",
    font_size = 20
)

In [ ]:

df = data_analysis_helper.merge_and_show_accuracy(
    df_list=[df_result_9],
    # threshold_type="mean",
    threshold_type="mean",
    chart_helper=chart_helper,
    title="Exp 9. Agent.",
    font_size = 20
)

# Footnote

- Copyright (c) 2026 Ivan Liu. Licensed under CC BY 4.0.
- Last Update: 2025/11/24
- Env: env51122
- References